
# Strava GPX — Project-Based Data Analysis + Interactive Web Project — Colab v14

**Choose a GPX → audit the raw data → learn pandas → engineer features → build a map → plot the ride → query it with SQL → export a website-ready ZIP**

This is the **Google Colab** version. It is deliberately both:

- a **working GPX analysis project**; and
- a **project-based way to learn pandas and plotting**.

The selected GPX is always treated as the source of truth.

## Time policy

1. If the GPX contains genuine point timestamps, use them.
2. If timestamps are missing, preserve that fact in the raw-data audit.
3. To keep the notebook simple and fully runnable, create a separate **placeholder analysis clock** beginning at **2000-01-01 00:00:00 UTC**, with one second added per track point.
4. Placeholder time is only a computational/indexing device. Speed, acceleration and other time-derived values are labelled as **placeholder-time estimates**, not genuine measurements.
5. If a later GPX contains real timestamps, the notebook automatically uses the real timing instead.

The final cell automatically creates a complete ZIP containing the processed data, figures, map, SQLite database, JSON/GeoJSON, reports and browser project.


In [ ]:

# FIRST EXECUTABLE CELL — FIND GPX FILES IN GOOGLE DRIVE QUICKLY
#
# This uses the Google Drive API to search by filename instead of recursively
# walking every folder in mounted My Drive. That is much faster on large Drives.
#
# Choose a GPX by number, or enter U to upload one from your computer.

from pathlib import Path
from google.colab import auth, files
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

BASE_DIR = Path("/content").resolve()
PROJECT_DIR = BASE_DIR

print("Authorising Google Drive search...")
auth.authenticate_user()
creds, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=creds)

print("Searching Google Drive for GPX files...")

gpx_items = []
page_token = None

while True:
    response = drive_service.files().list(
        q="trashed = false and name contains '.gpx'",
        spaces="drive",
        fields="nextPageToken, files(id,name,mimeType,modifiedTime)",
        pageSize=1000,
        pageToken=page_token,
        orderBy="modifiedTime desc"
    ).execute()

    for item in response.get("files", []):
        if item.get("name", "").lower().endswith(".gpx"):
            gpx_items.append(item)

    page_token = response.get("nextPageToken")
    if not page_token:
        break

if gpx_items:
    print(f"Found {len(gpx_items)} GPX file(s):")
    print()
    for i, item in enumerate(gpx_items, start=1):
        modified = item.get("modifiedTime", "")
        print(f"{i:>3}. {item['name']}   {modified}")

    print()
    choice = input(
        "Enter the number of the GPX to analyse, or U to upload one from this computer: "
    ).strip()

    if choice.lower() == "u":
        uploaded = files.upload()
        gpx_names = [name for name in uploaded if name.lower().endswith(".gpx")]
        if not gpx_names:
            raise RuntimeError("No .gpx file was uploaded.")
        GPX_PATH = BASE_DIR / gpx_names[0]
    else:
        try:
            selected_index = int(choice) - 1
            selected = gpx_items[selected_index]
        except (ValueError, IndexError):
            raise RuntimeError(
                "Invalid selection. Rerun this first cell and enter a valid GPX number."
            )

        # Download the selected Drive GPX into the Colab runtime.
        GPX_PATH = BASE_DIR / selected["name"]
        request = drive_service.files().get_media(fileId=selected["id"])
        with io.FileIO(GPX_PATH, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()

        print("Downloaded selected GPX from Drive:", GPX_PATH)
else:
    print("No GPX files were found in Google Drive.")
    print("Upload one from this computer instead.")
    uploaded = files.upload()
    gpx_names = [name for name in uploaded if name.lower().endswith(".gpx")]
    if not gpx_names:
        raise RuntimeError("No .gpx file was uploaded.")
    GPX_PATH = BASE_DIR / gpx_names[0]

ACTIVITY_TITLE = GPX_PATH.stem.replace("_", " ")
ACTIVITY_SLUG = "".join(
    ch if ch.isalnum() else "_" for ch in GPX_PATH.stem
).strip("_") or "activity"

EXPECTED_OUTPUT_DIR = PROJECT_DIR / "outputs" / f"{ACTIVITY_SLUG}_analysis"
EXPECTED_ZIP_PATH = PROJECT_DIR / "outputs" / f"{ACTIVITY_SLUG}_Ultimate_Web_Project_Bundle.zip"

print()
print("SELECTED GPX :", GPX_PATH)
print("OUTPUT FOLDER:", EXPECTED_OUTPUT_DIR)
print("ZIP WILL BE  :", EXPECTED_ZIP_PATH)


In [ ]:

# COLAB PACKAGE CHECK
# Colab normally has these already. Install only if something is missing.

import sys
import subprocess
import importlib.util

CORE_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "folium": "folium",
    "branca": "branca",
    "plotly": "plotly",
}

missing_core = [
    pip_name
    for module_name, pip_name in CORE_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_core:
    print("Installing:", ", ".join(missing_core))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_core]
    )

print("GPX already selected:", GPX_PATH.name)
print("Core analysis packages ready.")



# Get your GPX file from Strava

For your own Strava activity:

1. Sign in to **Strava.com** in a browser.
2. Open your activity.
3. Open the **More / ellipsis (`⋯`)** menu.
4. Choose **Export GPX**.
5. Save the `.gpx` anywhere on your computer.
6. Run the first Colab code cell and upload it.

If genuine point timestamps exist, the notebook uses them. If they are absent, the source missingness is preserved and the clearly labelled zero-origin placeholder clock is used for the teaching pipeline.



## GPX selected

The GPX was uploaded in the **first executable cell**, before pandas, NumPy, Matplotlib or Folium were loaded.

Its filename controls the analysis folder and final ZIP filename.



## Colab workflow

1. Run the **first executable cell**.
2. Approve Google Drive access when Colab asks.
3. The notebook searches Google Drive directly for filenames ending in `.gpx`.
4. Enter the number of the ride you want, or enter **U** to upload one from your computer.
5. The selected GPX is copied into the Colab runtime.
6. Run the rest of the notebook.
7. The GPX filename determines the analysis folder and ZIP filename.
8. The final cell builds and downloads the complete ZIP automatically.

This version deliberately avoids a recursive `MyDrive.rglob(...)` scan, because traversing a large mounted Drive can be very slow.



## Project-based learning track — learn pandas and plotting by building something real

Rather than learning isolated commands, each pandas or plotting idea solves a visible problem in the ride.

| Project question | pandas / plotting skill |
|---|---|
| What is actually inside the GPX? | `DataFrame`, `.head()`, `.tail()`, `.sample()`, `.dtypes` |
| Is anything missing or invalid? | `.isna()`, `.sum()`, boolean masks, `.between()` |
| What does a typical value look like? | `.describe()`, quantiles, median |
| How far did the rider travel? | vectorised columns, `.shift()`, `.cumsum()` |
| How noisy is elevation? | `.rolling()`, `.median()`, `.mean()` |
| How do we compare parts of the ride? | `.groupby()`, `.agg()` |
| How do we explain the data visually? | Matplotlib line plots, scatter plots, histograms, dual axes |
| How do we make the result reusable? | CSV, JSON, GeoJSON, SQLite / SQL, HTML |

**Learning rule:** whenever a new pandas operation appears, read the line as a transformation:

**input columns → operation → new column / summary → visual or decision**

That is the central data-analysis habit this project is designed to teach.


## Imports and output folders


In [ ]:
import json, math, shutil, sqlite3, sys, xml.etree.ElementTree as ET
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
import branca.colormap as bcm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from folium.plugins import Fullscreen, MousePosition
from IPython.display import display, HTML


# ------------------------------------------------------------
# COLAB FOLLOW-EXECUTION HELPER
# ------------------------------------------------------------
# Best effort only: asks the Colab front end to keep the currently executing
# cell near the centre of the screen during Run all.
try:
    from google.colab import output as _colab_output

    def follow_execution():
        try:
            _colab_output.eval_js("""
            (() => {
              const cells = [...document.querySelectorAll('colab-cell')];
              const running = cells.find(c =>
                c.hasAttribute('running') ||
                c.hasAttribute('executing') ||
                c.classList.contains('running') ||
                c.querySelector('[aria-label*="running" i],[title*="running" i]')
              );
              if (running) {
                running.scrollIntoView({behavior:'smooth', block:'center'});
                return true;
              }
              return false;
            })()
            """)
        except Exception:
            pass
except Exception:
    def follow_execution():
        pass

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

if GPX_PATH is None:
    raise RuntimeError("No GPX selected. Go back to the GPX selector cell and choose a file first.")

ACTIVITY_TITLE = GPX_PATH.stem.replace("_", " ")
ACTIVITY_SLUG = "".join(ch if ch.isalnum() else "_" for ch in GPX_PATH.stem).strip("_") or "activity"
OUTPUT_DIR = PROJECT_DIR / "outputs" / f"{ACTIVITY_SLUG}_analysis"
DATA_DIR = OUTPUT_DIR / "data"
FIGURES_DIR = OUTPUT_DIR / "figures"
MAPS_DIR = OUTPUT_DIR / "maps"
JSON_DIR = OUTPUT_DIR / "json"
SQL_DIR = OUTPUT_DIR / "sql"
WEBSITE_DIR = OUTPUT_DIR / "website"
REPORT_DIR = OUTPUT_DIR / "report"
SOURCE_DIR = OUTPUT_DIR / "source"

for d in [DATA_DIR, FIGURES_DIR, MAPS_DIR, JSON_DIR, SQL_DIR, WEBSITE_DIR, REPORT_DIR, SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Activity:", ACTIVITY_TITLE)
print("GPX:", GPX_PATH)
EXPECTED_ZIP_PATH = PROJECT_DIR / "outputs" / f"{ACTIVITY_SLUG}_Ultimate_Web_Project_Bundle.zip"
print("Output folder:", OUTPUT_DIR)
print("ZIP will be:", EXPECTED_ZIP_PATH)


## Part B — Parse the GPX + Audit the Raw Source

A GPX file is XML. The parser below deliberately strips XML namespaces from tag names, because `<time>` and `<ele>` can otherwise be missed by a narrow parser.

The EDA starts **before** calculating speed or gradient. We first establish what data actually exists.


In [ ]:
follow_execution()

def local_name(tag):
    return tag.split("}")[-1].lower()

tree = ET.parse(GPX_PATH)
root = tree.getroot()

# Read a little GPX-level metadata where available.
gpx_metadata = {
    "source_file": GPX_PATH.name,
    "root_tag": local_name(root.tag),
    "creator": root.attrib.get("creator"),
    "version": root.attrib.get("version"),
}

for element in root.iter():
    name = local_name(element.tag)
    if name == "name" and element.text and "activity_name" not in gpx_metadata:
        gpx_metadata["activity_name"] = element.text.strip()
        break

rows = []
for el in root.iter():
    if local_name(el.tag) != "trkpt":
        continue

    row = {
        "point_id": len(rows) + 1,
        "latitude": float(el.attrib["lat"]),
        "longitude": float(el.attrib["lon"]),
        "elevation_m": np.nan,
        "timestamp_raw": None,
    }

    for child in el:
        name = local_name(child.tag)
        text = child.text.strip() if child.text else ""

        if name == "ele" and text:
            row["elevation_m"] = float(text)
        elif name == "time" and text:
            row["timestamp_raw"] = text

    rows.append(row)

if not rows:
    raise ValueError("No GPX track points found.")

raw_df = pd.DataFrame(rows)
raw_df["time"] = pd.to_datetime(
    raw_df["timestamp_raw"],
    errors="coerce",
    utc=True
)

# Preserve the parsed GPX time exactly as received before any analysis fallback.
raw_df["gpx_time_original"] = raw_df["time"].copy()

usable_timestamps = int(raw_df["time"].notna().sum())
HAS_REAL_TIME = bool(
    len(raw_df) > 1
    and usable_timestamps == len(raw_df)
    and raw_df["time"].nunique() > 1
)

print("GPX metadata")
display(pd.DataFrame([gpx_metadata]).T.rename(columns={0:"value"}))

print("\nShape:", raw_df.shape)
print("Track points:", len(raw_df))
print("Usable timestamps:", usable_timestamps)
print("Complete real GPX time series available:", HAS_REAL_TIME)
print("Usable elevations:", int(raw_df["elevation_m"].notna().sum()))

print("\nHeaders")
for i, col in enumerate(raw_df.columns, start=1):
    print(f"{i:>2}. {col}")

print("\nFirst 8 rows")
display(raw_df.head(8))
print("Last 8 rows")
display(raw_df.tail(8))
print("Random sample")
display(raw_df.sample(min(8, len(raw_df)), random_state=42))

dtype_table = pd.DataFrame({
    "column": raw_df.columns,
    "dtype": [str(raw_df[c].dtype) for c in raw_df.columns],
    "non_null": [int(raw_df[c].notna().sum()) for c in raw_df.columns],
    "missing": [int(raw_df[c].isna().sum()) for c in raw_df.columns],
})
display(dtype_table)

missing_table = pd.DataFrame({
    "column": raw_df.columns,
    "missing_count": raw_df.isna().sum().values,
    "missing_percent": (raw_df.isna().mean().values * 100).round(2),
})
display(missing_table)

duplicate_rows = int(raw_df.duplicated().sum())

coordinate_checks = pd.DataFrame({
    "check": [
        "Exact duplicate raw rows",
        "Invalid latitude",
        "Invalid longitude",
        "Missing latitude",
        "Missing longitude",
        "Missing elevation",
        "Missing / unusable timestamp",
    ],
    "count": [
        duplicate_rows,
        int((~raw_df["latitude"].between(-90, 90)).sum()),
        int((~raw_df["longitude"].between(-180, 180)).sum()),
        int(raw_df["latitude"].isna().sum()),
        int(raw_df["longitude"].isna().sum()),
        int(raw_df["elevation_m"].isna().sum()),
        int(raw_df["time"].isna().sum()),
    ],
})
display(coordinate_checks)

raw_stats = raw_df[
    ["latitude", "longitude", "elevation_m"]
].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
).T

display(raw_stats)

raw_dictionary = pd.DataFrame([
    ["point_id", "integer", "Sequential GPX track-point number", "Raw"],
    ["latitude", "float", "Latitude in decimal degrees", "Raw"],
    ["longitude", "float", "Longitude in decimal degrees", "Raw"],
    ["elevation_m", "float", "Recorded elevation in metres", "Raw"],
    ["timestamp_raw", "text", "Original GPX timestamp text", "Raw"],
    ["time", "datetime", "Parsed UTC timestamp used during initial audit", "Parsed"],
    ["gpx_time_original", "datetime", "Original parsed GPX time preserved for provenance", "Parsed / preserved"],
], columns=["column", "type", "meaning", "source"])

display(raw_dictionary)

raw_stats.to_csv(DATA_DIR / "raw_descriptive_statistics.csv")
raw_dictionary.to_csv(DATA_DIR / "data_dictionary_raw.csv", index=False)

if HAS_REAL_TIME:
    print("Raw timestamp status: READY — genuine GPX timestamps found.")
else:
    print("Raw timestamp status: STRIPPED / MISSING — no complete point timestamp series exists in this GPX.")
    print("A separate reconstructed analysis clock will be created from the known Strava ride metadata in the next section.")



## Time handling — genuine GPX time or a zero-origin placeholder

The raw GPX audit above decides which path the notebook takes.

### State 1 — genuine timestamps are present

The notebook uses the recorded times exactly as supplied.

### State 2 — timestamps are absent

The raw missing values remain recorded as missing. For computational simplicity, a **separate analysis clock** is created:

- origin: **2000-01-01 00:00:00 UTC**
- step: **1 second per GPX point**

This is the datetime equivalent of filling the analysis clock from zero. It keeps one notebook fully runnable without asking the user for extra metadata.

**Important:** placeholder time is not recovered Strava time. Any speed, velocity or acceleration based on it is a teaching/example estimate and is labelled accordingly.


In [ ]:
follow_execution()


# ------------------------------------------------------------
# TIME SOURCE
# ------------------------------------------------------------

PLACEHOLDER_START_UTC = pd.Timestamp("2000-01-01T00:00:00Z")
PLACEHOLDER_STEP_SECONDS = 1.0

TIME_IS_SYNTHETIC = not HAS_REAL_TIME

if HAS_REAL_TIME:
    HAS_TIME = True
    TIME_SOURCE = "genuine_gpx_timestamps"
    raw_df["analysis_time"] = raw_df["gpx_time_original"]
    raw_df["analysis_time_source"] = "genuine GPX timestamp"
    TIME_AXIS_LABEL = "Elapsed time (minutes)"
    TIME_SERIES_NOTE = (
        "Time-dependent measures use genuine point timestamps from the GPX."
    )
else:
    HAS_TIME = True
    TIME_SOURCE = "zero_origin_placeholder_1_second_per_point"

    if len(raw_df) < 2:
        raise ValueError("At least two GPX points are required for the analysis clock.")

    raw_df["analysis_time"] = (
        PLACEHOLDER_START_UTC
        + pd.to_timedelta(
            np.arange(len(raw_df)) * PLACEHOLDER_STEP_SECONDS,
            unit="s"
        )
    )
    raw_df["analysis_time_source"] = (
        "placeholder: 2000-01-01 00:00:00 UTC + 1 second per point"
    )

    TIME_AXIS_LABEL = "Placeholder time (minutes)"
    TIME_SERIES_NOTE = (
        "The GPX contains no usable point timestamps. The raw timestamp fields remain "
        "missing. A separate zero-origin placeholder clock beginning at 2000-01-01 "
        "00:00:00 UTC advances by one second per GPX point so the full pandas and "
        "plotting pipeline can run. Time-derived speed, velocity and acceleration are "
        "therefore placeholder-time estimates, not genuine measurements."
    )

# Downstream feature engineering uses one consistent analysis column.
raw_df["time"] = raw_df["analysis_time"]

TIME_PROVENANCE = {
    "has_real_gpx_timestamps": bool(HAS_REAL_TIME),
    "time_available_for_computation": bool(HAS_TIME),
    "time_is_placeholder": bool(TIME_IS_SYNTHETIC),
    "time_source": TIME_SOURCE,
    "placeholder_start_utc": str(PLACEHOLDER_START_UTC) if TIME_IS_SYNTHETIC else None,
    "placeholder_step_seconds": PLACEHOLDER_STEP_SECONDS if TIME_IS_SYNTHETIC else None,
    "note": TIME_SERIES_NOTE,
}

display(HTML(
    "<div style='padding:12px 14px;border:1px solid #dbe2ea;border-radius:10px;"
    "margin:8px 0 12px;line-height:1.5'>"
    f"<strong>Time source:</strong> {TIME_SOURCE}<br>"
    f"<strong>Genuine GPX timestamps:</strong> {HAS_REAL_TIME}<br>"
    f"<strong>Placeholder clock:</strong> {TIME_IS_SYNTHETIC}<br><br>"
    f"{TIME_SERIES_NOTE}"
    "</div>"
))

(JSON_DIR / "time_provenance.json").write_text(
    json.dumps(TIME_PROVENANCE, indent=2),
    encoding="utf-8"
)


## Part C — Feature Engineering

The GPX gives us recorded observations. The useful analytical variables are created from them.

### Raw → engineered

**Latitude + longitude** → segment distance, cumulative distance, displacement from the start, bearing  
**Elevation** → smoothed elevation, gain, loss, cumulative climbing, gradient  
**Timestamps (when present)** → elapsed time, speed, velocity components, acceleration, vertical speed, moving/stopped state

This distinction between **source fields** and **engineered fields** is central to the project.


### pandas checkpoint

Watch how the analysis moves from a small number of source columns to useful derived columns. Key operations here include:

- `.shift(1)` to compare each point with the previous point;
- vectorised NumPy/pandas arithmetic instead of row-by-row loops;
- `.cumsum()` to build cumulative distance;
- `.rolling()` to smooth noisy measurements;
- `.diff()` to calculate change.

Try reading each assignment as: **create a new column from existing columns**.


In [ ]:
follow_execution()

EARTH_RADIUS_M = 6_371_000.0
STOP_SPEED_KMH = 1.0
HARD_SPEED_LIMIT_KMH = 80.0


def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance between latitude/longitude points."""
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return EARTH_RADIUS_M * 2 * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )


def bearing_deg(lat1, lon1, lat2, lon2):
    """Compass bearing from point 1 to point 2."""
    lat1r = np.radians(lat1)
    lat2r = np.radians(lat2)
    dlon = np.radians(lon2 - lon1)

    x = np.sin(dlon) * np.cos(lat2r)
    y = (
        np.cos(lat1r) * np.sin(lat2r)
        - np.sin(lat1r) * np.cos(lat2r) * np.cos(dlon)
    )

    return (np.degrees(np.arctan2(x, y)) + 360) % 360


# Preserve original GPX point order when time is unavailable.
df = raw_df.copy()
if HAS_TIME:
    df = df.sort_values("time")
df = df.reset_index(drop=True)

lat_prev = df["latitude"].shift(1)
lon_prev = df["longitude"].shift(1)

# Distance travelled between consecutive GPS points.
df["segment_distance_m"] = haversine_m(
    lat_prev,
    lon_prev,
    df["latitude"],
    df["longitude"]
)
df.loc[0, "segment_distance_m"] = 0.0

df["bearing_deg"] = bearing_deg(
    lat_prev,
    lon_prev,
    df["latitude"],
    df["longitude"]
)
df.loc[0, "bearing_deg"] = np.nan

# Straight-line displacement magnitude from the starting point.
df["displacement_from_start_m"] = haversine_m(
    df["latitude"].iloc[0],
    df["longitude"].iloc[0],
    df["latitude"],
    df["longitude"]
)

# Raw cumulative distance is always non-decreasing.
df["cumulative_distance_raw_km"] = (
    df["segment_distance_m"]
    .fillna(0)
    .clip(lower=0)
    .cumsum()
    / 1000.0
)

# Elevation smoothing before gradient/ascent calculations.
df["elevation_smooth_m"] = (
    df["elevation_m"]
    .interpolate(limit_direction="both")
    .rolling(21, center=True, min_periods=1)
    .median()
    .rolling(15, center=True, min_periods=1)
    .mean()
)

df["elevation_change_m"] = df["elevation_smooth_m"].diff()
df["elevation_gain_m"] = df["elevation_change_m"].clip(lower=0).fillna(0)
df["elevation_loss_m"] = (-df["elevation_change_m"].clip(upper=0)).fillna(0)
df["cumulative_elevation_gain_m"] = df["elevation_gain_m"].cumsum()
df["cumulative_elevation_loss_m"] = df["elevation_loss_m"].cumsum()

# A second ascent estimate is calculated from a fixed-distance resampled profile.
# This reduces the tendency of tiny point-to-point elevation noise to inflate
# total ascent. It is kept separate from the pointwise cumulative-gain series.
_distance_m_for_elevation = df["cumulative_distance_raw_km"].to_numpy() * 1000.0
_elevation_for_profile = df["elevation_smooth_m"].to_numpy()

if len(df) > 1 and np.nanmax(_distance_m_for_elevation) > 0:
    ELEVATION_PROFILE_STEP_M = 50.0
    _grid_m = np.arange(0.0, np.nanmax(_distance_m_for_elevation) + ELEVATION_PROFILE_STEP_M, ELEVATION_PROFILE_STEP_M)
    _grid_elev = np.interp(_grid_m, _distance_m_for_elevation, _elevation_for_profile)
    _grid_elev = pd.Series(_grid_elev).rolling(5, center=True, min_periods=1).median().to_numpy()
    _grid_diff = np.diff(_grid_elev)
    elevation_gain_profile_m = float(np.clip(_grid_diff, 0, None).sum())
    elevation_loss_profile_m = float((-np.clip(_grid_diff, None, 0)).sum())
else:
    ELEVATION_PROFILE_STEP_M = 50.0
    elevation_gain_profile_m = np.nan
    elevation_loss_profile_m = np.nan

# Time-dependent engineering is conditional.
if HAS_TIME:
    df["delta_time_s"] = df["time"].diff().dt.total_seconds()
    df["elapsed_time_s"] = (
        df["time"] - df["time"].iloc[0]
    ).dt.total_seconds()
    df["elapsed_time_min"] = df["elapsed_time_s"] / 60.0

    df["speed_m_s_raw"] = df["segment_distance_m"] / df["delta_time_s"]
    df["speed_kmh_raw"] = df["speed_m_s_raw"] * 3.6

    df["nonpositive_time_flag"] = (
        df["delta_time_s"].notna()
        & (df["delta_time_s"] <= 0)
    )

    valid_speed = df["speed_kmh_raw"].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    q1 = valid_speed.quantile(0.25)
    q3 = valid_speed.quantile(0.75)
    iqr = q3 - q1
    iqr_upper = q3 + 3.0 * iqr

    speed_screen_limit = min(
        HARD_SPEED_LIMIT_KMH,
        iqr_upper if np.isfinite(iqr_upper) else HARD_SPEED_LIMIT_KMH
    )

    df["speed_outlier_flag"] = (
        (df["speed_kmh_raw"] > speed_screen_limit)
        | df["nonpositive_time_flag"]
    )

    speed_clean = (
        df["speed_kmh_raw"]
        .mask(df["speed_outlier_flag"])
        .interpolate(limit_direction="both")
    )

    df["speed_smooth_kmh"] = (
        speed_clean
        .rolling(9, center=True, min_periods=1)
        .median()
        .rolling(7, center=True, min_periods=1)
        .mean()
    )

    df["speed_smooth_m_s"] = df["speed_smooth_kmh"] / 3.6

    # Below the stop threshold, tiny GPS jitter should not add travelled distance.
    df["analysis_segment_distance_m"] = (
        df["segment_distance_m"].fillna(0).clip(lower=0)
    )
    df.loc[
        (df["speed_smooth_kmh"] < STOP_SPEED_KMH)
        | df["speed_outlier_flag"],
        "analysis_segment_distance_m"
    ] = 0.0

    df["cumulative_distance_km"] = (
        df["analysis_segment_distance_m"].cumsum() / 1000.0
    )

    # Speed is the magnitude of velocity. Bearing gives direction.
    bearing_rad = np.radians(df["bearing_deg"])
    df["velocity_east_m_s"] = df["speed_smooth_m_s"] * np.sin(bearing_rad)
    df["velocity_north_m_s"] = df["speed_smooth_m_s"] * np.cos(bearing_rad)

    # Scalar/tangential acceleration from change in smoothed speed.
    df["acceleration_m_s2"] = (
        df["speed_smooth_m_s"].diff() / df["delta_time_s"]
    ).replace([np.inf, -np.inf], np.nan)

    acc_lo, acc_hi = df["acceleration_m_s2"].quantile([0.01, 0.99])
    df["acceleration_smooth_m_s2"] = (
        df["acceleration_m_s2"]
        .clip(acc_lo, acc_hi)
        .rolling(9, center=True, min_periods=1)
        .median()
    )

    # Component accelerations show changes in the velocity vector.
    df["acceleration_east_m_s2"] = (
        df["velocity_east_m_s"].diff() / df["delta_time_s"]
    ).replace([np.inf, -np.inf], np.nan)
    df["acceleration_north_m_s2"] = (
        df["velocity_north_m_s"].diff() / df["delta_time_s"]
    ).replace([np.inf, -np.inf], np.nan)

    df["vertical_speed_m_s"] = (
        df["elevation_smooth_m"].diff() / df["delta_time_s"]
    ).replace([np.inf, -np.inf], np.nan)

    df["is_moving"] = df["speed_smooth_kmh"] >= STOP_SPEED_KMH

else:
    speed_screen_limit = np.nan
    df["delta_time_s"] = np.nan
    df["elapsed_time_s"] = np.nan
    df["elapsed_time_min"] = np.nan
    df["speed_m_s_raw"] = np.nan
    df["speed_kmh_raw"] = np.nan
    df["nonpositive_time_flag"] = False
    df["speed_outlier_flag"] = False
    df["speed_smooth_kmh"] = np.nan
    df["speed_smooth_m_s"] = np.nan
    df["analysis_segment_distance_m"] = df["segment_distance_m"].fillna(0).clip(lower=0)
    df["cumulative_distance_km"] = df["cumulative_distance_raw_km"]
    df["velocity_east_m_s"] = np.nan
    df["velocity_north_m_s"] = np.nan
    df["acceleration_m_s2"] = np.nan
    df["acceleration_smooth_m_s2"] = np.nan
    df["acceleration_east_m_s2"] = np.nan
    df["acceleration_north_m_s2"] = np.nan
    df["vertical_speed_m_s"] = np.nan
    df["is_moving"] = np.nan

# Gradient uses distance + elevation, so it is available even without timestamps.
GRADE_LAG = max(5, min(25, len(df) // 200 if len(df) >= 200 else 5))
rise = df["elevation_smooth_m"].diff(GRADE_LAG)
run_m = df["cumulative_distance_km"].diff(GRADE_LAG) * 1000.0

df["grade_smooth_pct"] = np.where(
    run_m.abs() >= 10,
    (rise / run_m) * 100.0,
    np.nan
)

# Central requested invariant: cumulative travelled distance can only rise or stay flat.
assert (
    df["cumulative_distance_km"]
    .diff()
    .fillna(0)
    >= -1e-12
).all()

engineered_dictionary = pd.DataFrame([
    ["segment_distance_m", "m", "Haversine distance from previous GPS point"],
    ["cumulative_distance_km", "km", "Cumulative travelled distance; never decreases"],
    ["displacement_from_start_m", "m", "Straight-line displacement magnitude from the start"],
    ["bearing_deg", "degrees", "Direction of travel"],
    ["elevation_smooth_m", "m", "Smoothed elevation"],
    ["elevation_gain_m", "m", "Positive elevation change"],
    ["cumulative_elevation_gain_m", "m", "Accumulated climbing"],
    ["grade_smooth_pct", "%", "Smoothed rise/run gradient"],
    ["elapsed_time_min", "minutes", "Elapsed / reconstructed moving time from the documented time source"],
    ["speed_smooth_kmh", "km/h", "Smoothed speed / velocity magnitude"],
    ["velocity_east_m_s", "m/s", "East component of velocity"],
    ["velocity_north_m_s", "m/s", "North component of velocity"],
    ["acceleration_smooth_m_s2", "m/s²", "Approximate tangential acceleration"],
    ["vertical_speed_m_s", "m/s", "Rate of elevation change"],
    ["is_moving", "boolean", f"Speed at least {STOP_SPEED_KMH} km/h"],
    ["speed_outlier_flag", "boolean", "Segment screened for investigation"],
], columns=["column", "unit", "meaning"])

print("Feature engineering complete.")
print("Distance monotonicity check: PASSED")
print("Time-series mode: ON")
print("Time source:", TIME_SOURCE)
print("Synthetic time:", TIME_IS_SYNTHETIC)
if HAS_TIME:
    print("Speed screening limit:", round(speed_screen_limit, 1), "km/h")

display(engineered_dictionary)
display(df.head(12))

engineered_dictionary.to_csv(
    DATA_DIR / "data_dictionary_engineered.csv",
    index=False
)


## Part D — Descriptive Statistics + Story of the Ride

This section produces both conventional EDA statistics and the small number of memorable facts that later become the narrative of the website.


In [ ]:
follow_execution()

# Sampling statistics describe the analysis clock. If TIME_IS_SYNTHETIC is True,
# the interval is a reconstruction assumption rather than observed device sampling.
if HAS_TIME:
    sampling = df["delta_time_s"].dropna()
    sampling = sampling[sampling > 0]
    sampling_summary = pd.Series({
        "count": sampling.count(),
        "mean_s": sampling.mean(),
        "median_s": sampling.median(),
        "std_s": sampling.std(),
        "min_s": sampling.min(),
        "p05_s": sampling.quantile(0.05),
        "p95_s": sampling.quantile(0.95),
        "max_s": sampling.max(),
    })
    display(sampling_summary.to_frame("value"))
else:
    sampling = pd.Series(dtype=float)
    sampling_summary = pd.Series(dtype=float)

analysis_columns = [
    "segment_distance_m",
    "cumulative_distance_km",
    "displacement_from_start_m",
    "elevation_m",
    "elevation_smooth_m",
    "grade_smooth_pct",
    "bearing_deg",
]
if HAS_TIME:
    analysis_columns += [
        "delta_time_s",
        "speed_kmh_raw",
        "speed_smooth_kmh",
        "velocity_east_m_s",
        "velocity_north_m_s",
        "acceleration_smooth_m_s2",
        "vertical_speed_m_s",
    ]

descriptive_stats = df[analysis_columns].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
).T

display(descriptive_stats)
descriptive_stats.to_csv(DATA_DIR / "engineered_descriptive_statistics.csv")

distance_km = float(df["cumulative_distance_km"].iloc[-1])
raw_distance_km = float(df["cumulative_distance_raw_km"].iloc[-1])

duration_s = float(df["elapsed_time_s"].iloc[-1]) if HAS_TIME else np.nan
moving_time_s = (
    float(
        df.loc[df["is_moving"] == True, "delta_time_s"]
        .clip(lower=0)
        .fillna(0)
        .sum()
    )
    if HAS_TIME else np.nan
)

avg_speed = (
    distance_km / (duration_s / 3600.0)
    if HAS_TIME and duration_s > 0 else np.nan
)
avg_moving_speed = (
    distance_km / (moving_time_s / 3600.0)
    if HAS_TIME and moving_time_s > 0 else np.nan
)

highest_idx = df["elevation_smooth_m"].idxmax()
highest = df.loc[highest_idx]
steep_up_idx = df["grade_smooth_pct"].idxmax()
steep_down_idx = df["grade_smooth_pct"].idxmin()
steep_up = df.loc[steep_up_idx]
steep_down = df.loc[steep_down_idx]

if HAS_TIME:
    fastest_idx = df["speed_smooth_kmh"].idxmax()
    fastest = df.loc[fastest_idx]
    speed_grade_pair = df[["speed_smooth_kmh", "grade_smooth_pct"]].dropna()
    speed_grade_corr = float(speed_grade_pair.corr().iloc[0, 1]) if len(speed_grade_pair) > 2 else np.nan
else:
    fastest = None
    speed_grade_corr = np.nan

ride_summary_rows = [
    ["Track points", len(df), "points"],
    ["Raw GPS distance", raw_distance_km, "km"],
    ["Cumulative analysed distance", distance_km, "km"],
    ["Pointwise smoothed elevation gain", df["elevation_gain_m"].sum(), "m"],
    ["50 m resampled-profile elevation gain", elevation_gain_profile_m, "m"],
    ["Elevation loss", df["elevation_loss_m"].sum(), "m"],
    ["Minimum elevation", df["elevation_smooth_m"].min(), "m"],
    ["Maximum elevation", df["elevation_smooth_m"].max(), "m"],
    ["Maximum displacement from start", df["displacement_from_start_m"].max() / 1000.0, "km"],
]

if HAS_TIME:
    ride_summary_rows += [
        ["Recorded duration" if HAS_REAL_TIME else "Placeholder clock span", duration_s / 60.0, "minutes"],
        ["Recorded moving time" if HAS_REAL_TIME else "Placeholder moving span", moving_time_s / 60.0, "minutes"],
        ["Overall average speed" if HAS_REAL_TIME else "Placeholder-time average speed", avg_speed, "km/h"],
        ["Average moving speed" if HAS_REAL_TIME else "Placeholder-time moving average", avg_moving_speed, "km/h"],
        ["Maximum smoothed speed" if HAS_REAL_TIME else "Placeholder-time maximum speed", df["speed_smooth_kmh"].max(), "km/h"],
        ["Median smoothed speed", df["speed_smooth_kmh"].median(), "km/h"],
        ["Median sampling interval", sampling.median(), "seconds"],
        ["Speed-gradient Pearson correlation", speed_grade_corr, "r"],
    ]

ride_summary = pd.DataFrame(
    ride_summary_rows,
    columns=["metric", "value", "unit"]
)

display(ride_summary)
ride_summary.to_csv(DATA_DIR / "ride_summary.csv", index=False)

data_quality = pd.DataFrame([
    ["Rows / track points", len(df), "Information"],
    ["Exact duplicate raw rows", duplicate_rows, "Check"],
    ["Missing latitude", int(raw_df["latitude"].isna().sum()), "Check"],
    ["Missing longitude", int(raw_df["longitude"].isna().sum()), "Check"],
    ["Invalid latitude", int((~raw_df["latitude"].between(-90, 90)).sum()), "Check"],
    ["Invalid longitude", int((~raw_df["longitude"].between(-180, 180)).sum()), "Check"],
    ["Missing elevation", int(raw_df["elevation_m"].isna().sum()), "Check"],
    ["Missing original GPX timestamps", int(raw_df["gpx_time_original"].isna().sum()), "Raw-source check"],
    ["Non-positive time gaps", int(df["nonpositive_time_flag"].sum()) if HAS_TIME else 0, "Screening flag"],
    ["Speed outlier flags", int(df["speed_outlier_flag"].sum()) if HAS_TIME else 0, "Screening flag"],
], columns=["check", "count", "interpretation"])

display(data_quality)
data_quality.to_csv(DATA_DIR / "data_quality_summary.csv", index=False)

story_points = [
    f"The GPX contains {len(df):,} recorded track points.",
    f"The analysed route covers {distance_km:.2f} km.",
    f"The highest smoothed elevation is {highest['elevation_smooth_m']:.1f} m at about {highest['cumulative_distance_km']:.2f} km.",
    f"Pointwise smoothed cumulative gain is {df['elevation_gain_m'].sum():.0f} m; a 50 m resampled-profile estimate is {elevation_gain_profile_m:.0f} m.", 
    "Cumulative elevation gain only increases because it is total climbing accumulated so far; the elevation profile itself rises and falls.",
    f"The steepest screened uphill gradient is about {steep_up['grade_smooth_pct']:.1f}% and the steepest downhill gradient is about {steep_down['grade_smooth_pct']:.1f}%.",
]

if HAS_REAL_TIME:
    story_points += [
        f"The genuine recorded duration represented by the GPX is {duration_s/60:.1f} minutes.",
        f"Average moving speed is {avg_moving_speed:.1f} km/h.",
        f"Maximum smoothed speed is {fastest['speed_smooth_kmh']:.1f} km/h at about {fastest['elapsed_time_min']:.1f} minutes.",
        f"The speed-gradient Pearson correlation for valid points is {speed_grade_corr:.2f}; this is descriptive association, not proof of causation.",
    ]
else:
    story_points += [
        "The source GPX has no genuine point timestamps.",
        "A zero-origin one-second-per-point placeholder clock keeps the learning pipeline operational.",
        "Time-derived speed, velocity and acceleration are examples under that placeholder assumption and should not be interpreted as measured ride performance.",
    ]

print("\nSTORY OF THE RIDE")
for point in story_points:
    print("•", point)



## Map — one real interactive Leaflet map

There is **one map only**.

Google Colab displays the actual Folium/Leaflet map directly, and the same map is saved as HTML for the website bundle.

It uses a Esri World Street Map basemap, plus the GPX route, start and finish, highest point, kilometre markers, elevation colouring, and a speed layer only when genuine GPX timestamps are present.


> **Colab map fix:** this version does not request tiles directly from `tile.openstreetmap.org`, because those requests can be blocked with HTTP 403 in hosted notebook environments. The route is drawn over an Esri World Street Map basemap instead.


> **v5 fix:** the CARTO layer in v4 displayed an API-key-required watermark in Colab, so v5 uses Esri public map tiles instead.


> **v6 display fix:** Colab can compress Folium's automatic notebook representation into a shallow strip. This version explicitly fixes the single map at **720 px high** in both the Leaflet document and its Colab iframe, so it should display as a normal large map rather than a thin horizontal band.

> **v7 synchronized dashboard:** the Play button / master slider moves the bicycle along the map and simultaneously moves a vertical cursor across the elevation and gradient profiles. Time-based charts follow elapsed time; terrain charts follow cumulative distance, so every panel stays aligned to the same ride position.


> **v9 interactive graphs:** every main line graph now supports nearest-point hover tooltips. Hovering shows the graph value plus ride context such as elevation, gradient, speed and elapsed time. Clicking or tapping a graph selects that ride position and moves the master slider and map marker there. The Play button continues to animate the map and synchronized graph cursors.


In [ ]:
follow_execution()

map_df = df[[
    "latitude", "longitude", "cumulative_distance_km",
    "elevation_smooth_m", "grade_smooth_pct", "bearing_deg",
    "elapsed_time_min", "speed_smooth_kmh"
]].dropna(subset=["latitude", "longitude"]).copy()

centre = [
    float(map_df["latitude"].mean()),
    float(map_df["longitude"].mean())
]

m = folium.Map(
    location=centre,
    zoom_start=12,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
    width="100%",
    height=650,
)

# Use Esri public basemap tiles.
# This avoids the OpenStreetMap 403 block and the CARTO API-key watermark
# seen in this hosted Colab environment.
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri",
    name="Esri World Street Map",
    max_zoom=19,
    overlay=False,
    control=True,
    show=True,
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri",
    name="Esri World Topographic Map",
    max_zoom=19,
    overlay=False,
    control=True,
    show=False,
).add_to(m)

route = map_df[["latitude", "longitude"]].values.tolist()

base_route = folium.FeatureGroup(name="Route", show=True)
folium.PolyLine(
    route,
    weight=4,
    opacity=0.55,
    tooltip=ACTIVITY_TITLE
).add_to(base_route)
base_route.add_to(m)


def add_coloured_route(data, value_col, layer_name, caption, show=False):
    valid = data[value_col].replace([np.inf, -np.inf], np.nan).dropna()
    if valid.empty:
        return

    vmin = float(valid.quantile(0.02))
    vmax = float(valid.quantile(0.98))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        return

    scale = bcm.linear.viridis.scale(vmin, vmax)
    scale.caption = caption
    layer = folium.FeatureGroup(name=layer_name, show=show)
    chunk_size = max(3, len(data) // 350)

    for start in range(0, len(data) - 1, chunk_size):
        part = data.iloc[start:min(start + chunk_size + 1, len(data))]
        value = part[value_col].replace([np.inf, -np.inf], np.nan).median()
        if pd.isna(value):
            continue
        points = part[["latitude", "longitude"]].dropna().values.tolist()
        if len(points) < 2:
            continue
        folium.PolyLine(
            points,
            weight=7,
            opacity=0.88,
            color=scale(float(np.clip(value, vmin, vmax))),
            tooltip=f"{caption}: {value:.1f}"
        ).add_to(layer)

    layer.add_to(m)
    if show:
        scale.add_to(m)


add_coloured_route(
    map_df,
    "elevation_smooth_m",
    "Route coloured by elevation",
    "Elevation (m)",
    show=not HAS_REAL_TIME
)

if HAS_REAL_TIME:
    add_coloured_route(
        map_df,
        "speed_smooth_kmh",
        "Route coloured by speed",
        "Speed (km/h)",
        show=True
    )

folium.Marker(
    route[0],
    tooltip="Start",
    popup="Start",
    icon=folium.Icon(color="green", icon="play")
).add_to(m)

folium.Marker(
    route[-1],
    tooltip="Finish",
    popup=f"Finish — {distance_km:.2f} km",
    icon=folium.Icon(color="red", icon="stop")
).add_to(m)

folium.Marker(
    [highest["latitude"], highest["longitude"]],
    tooltip="Highest point",
    popup=(
        f"<b>Highest point</b><br>"
        f"Elevation: {highest['elevation_smooth_m']:.1f} m<br>"
        f"Distance: {highest['cumulative_distance_km']:.2f} km"
    ),
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(m)

if HAS_REAL_TIME and fastest is not None:
    folium.Marker(
        [fastest["latitude"], fastest["longitude"]],
        tooltip="Maximum smoothed speed",
        popup=(
            f"<b>Maximum smoothed speed</b><br>"
            f"Speed: {fastest['speed_smooth_kmh']:.1f} km/h<br>"
            f"Time: {fastest['elapsed_time_min']:.1f} min<br>"
            f"Gradient: {fastest['grade_smooth_pct']:.1f}%"
        ),
        icon=folium.Icon(color="orange", icon="info-sign")
    ).add_to(m)

km_layer = folium.FeatureGroup(name="Kilometre markers", show=False)
for km in range(1, int(np.floor(distance_km)) + 1):
    idx = (df["cumulative_distance_km"] - km).abs().idxmin()
    row = df.loc[idx]
    details = [
        f"<b>{km} km</b>",
        f"Elevation: {row['elevation_smooth_m']:.1f} m",
        f"Gradient: {row['grade_smooth_pct']:.1f}%",
    ]
    if HAS_TIME:
        details += [
            f"Time: {row['elapsed_time_min']:.1f} min",
            f"Speed: {row['speed_smooth_kmh']:.1f} km/h",
        ]
    folium.CircleMarker(
        [row["latitude"], row["longitude"]],
        radius=4,
        weight=1,
        fill=True,
        fill_opacity=0.95,
        tooltip=f"{km} km",
        popup="<br>".join(details)
    ).add_to(km_layer)
km_layer.add_to(m)

Fullscreen(position="topleft").add_to(m)
MousePosition(position="bottomright", separator=" | ", prefix="Lat / Lon:").add_to(m)
folium.LatLngPopup().add_to(m)
m.fit_bounds(route)
folium.LayerControl(collapsed=False).add_to(m)

MAP_PATH = MAPS_DIR / "ride_full_width_map.html"
m.save(MAP_PATH)

print("Saved:", MAP_PATH)

# ------------------------------------------------------------
# COLAB DISPLAY — FORCE A PROPER MAP HEIGHT
# ------------------------------------------------------------
# Colab's automatic Folium renderer can squash the map vertically even when
# Folium itself was created with height=650.  We therefore render the SAME
# Folium map into an explicit srcdoc iframe and set BOTH:
#   1) the Leaflet map div height, and
#   2) the outer iframe height.
#
# This is still one map, not a duplicate.

import html as _html

MAP_DISPLAY_HEIGHT_PX = 720
_map_id = m.get_name()
_map_document = m.get_root().render()

_force_height_css = f"""
<style>
html, body {{
    width: 100% !important;
    height: 100% !important;
    margin: 0 !important;
    padding: 0 !important;
}}
#{_map_id} {{
    width: 100% !important;
    height: {MAP_DISPLAY_HEIGHT_PX}px !important;
    min-height: {MAP_DISPLAY_HEIGHT_PX}px !important;
}}
</style>
"""

_map_document = _map_document.replace(
    "</head>",
    _force_height_css + "</head>",
    1
)

_map_srcdoc = _html.escape(_map_document, quote=True)

display(HTML(
    f"""
    <iframe
        srcdoc="{_map_srcdoc}"
        style="
            width:100%;
            height:{MAP_DISPLAY_HEIGHT_PX + 20}px;
            min-height:{MAP_DISPLAY_HEIGHT_PX + 20}px;
            border:1px solid #dbe2ea;
            border-radius:10px;
            display:block;
        "
        loading="eager">
    </iframe>
    """
))



## Interactive ride dashboard — inside Colab

This is separate from the static Matplotlib figure set later in the notebook.

You can now interact with the selected ride point in three ways:

- **drag the ride-position slider** to move the persistent dot through the ride;
- use **Play / Pause** to animate the same dot automatically;
- **hover over the lines** to inspect exact local values.

The selected point moves simultaneously on the route, elevation, gradient and speed/climbing panels.

After zooming or panning, click **↺ Reset zoom** below the dashboard. Plotly's own toolbar also remains visible, including its reset/home controls.

The same dashboard is exported as:

`website/interactive_ride_dashboard.html`

for the portfolio website.


In [ ]:

follow_execution()

# ------------------------------------------------------------
# PLOTLY INTERACTIVE RIDE DASHBOARD
# ------------------------------------------------------------

plot_df = df.reset_index(drop=True).copy()

if len(plot_df) < 2:
    raise RuntimeError("At least two track points are required for the interactive dashboard.")

# Keep the animation responsive while preserving the full line data for hover.
FRAME_COUNT = min(180, len(plot_df))
frame_indices = np.unique(
    np.linspace(0, len(plot_df) - 1, FRAME_COUNT, dtype=int)
)

has_speed = (
    "speed_smooth_kmh" in plot_df.columns
    and plot_df["speed_smooth_kmh"].notna().any()
)

if has_speed:
    panel4_values = plot_df["speed_smooth_kmh"]
    panel4_title = "Speed vs distance"
    panel4_y_title = "Speed (km/h)"
    panel4_name = "Speed"
else:
    panel4_values = plot_df["cumulative_elevation_gain_m"]
    panel4_title = "Cumulative climbing vs distance"
    panel4_y_title = "Elevation gain (m)"
    panel4_name = "Cumulative climbing"

speed_context = (
    plot_df["speed_smooth_kmh"].to_numpy()
    if "speed_smooth_kmh" in plot_df.columns
    else np.full(len(plot_df), np.nan)
)

elapsed_context = (
    plot_df["elapsed_time_min"].to_numpy()
    if "elapsed_time_min" in plot_df.columns
    else np.full(len(plot_df), np.nan)
)

custom = np.column_stack([
    plot_df["cumulative_distance_km"].to_numpy(),
    plot_df["elevation_smooth_m"].to_numpy(),
    plot_df["grade_smooth_pct"].to_numpy(),
    speed_context,
    elapsed_context,
])

fig_interactive = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Route position",
        "Elevation vs distance",
        "Gradient vs distance",
        panel4_title,
    ),
    horizontal_spacing=0.10,
    vertical_spacing=0.16,
)

# Trace 0 — route
fig_interactive.add_trace(
    go.Scatter(
        x=plot_df["longitude"],
        y=plot_df["latitude"],
        mode="lines",
        name="Route",
        customdata=custom,
        hovertemplate=(
            "Distance: %{customdata[0]:.2f} km<br>"
            "Elevation: %{customdata[1]:.1f} m<br>"
            "Gradient: %{customdata[2]:.1f}%<br>"
            "Speed: %{customdata[3]:.1f} km/h<br>"
            "Elapsed: %{customdata[4]:.1f} min"
            "<extra></extra>"
        ),
    ),
    row=1, col=1
)

# Trace 1 — moving route point
fig_interactive.add_trace(
    go.Scatter(
        x=[plot_df.loc[0, "longitude"]],
        y=[plot_df.loc[0, "latitude"]],
        mode="markers",
        marker=dict(size=13),
        name="Selected point",
        showlegend=False,
        hovertemplate="Selected ride point<extra></extra>",
    ),
    row=1, col=1
)

# Trace 2 — elevation
fig_interactive.add_trace(
    go.Scatter(
        x=plot_df["cumulative_distance_km"],
        y=plot_df["elevation_smooth_m"],
        mode="lines",
        name="Elevation",
        customdata=custom,
        hovertemplate=(
            "Distance: %{x:.2f} km<br>"
            "Elevation: %{y:.1f} m<br>"
            "Gradient: %{customdata[2]:.1f}%<br>"
            "Speed: %{customdata[3]:.1f} km/h"
            "<extra></extra>"
        ),
    ),
    row=1, col=2
)

# Trace 3 — moving elevation point
fig_interactive.add_trace(
    go.Scatter(
        x=[plot_df.loc[0, "cumulative_distance_km"]],
        y=[plot_df.loc[0, "elevation_smooth_m"]],
        mode="markers",
        marker=dict(size=11),
        showlegend=False,
        hovertemplate="Elevation: %{y:.1f} m<extra></extra>",
    ),
    row=1, col=2
)

# Trace 4 — gradient
fig_interactive.add_trace(
    go.Scatter(
        x=plot_df["cumulative_distance_km"],
        y=plot_df["grade_smooth_pct"],
        mode="lines",
        name="Gradient",
        customdata=custom,
        hovertemplate=(
            "Distance: %{x:.2f} km<br>"
            "Gradient: %{y:.2f}%<br>"
            "Elevation: %{customdata[1]:.1f} m<br>"
            "Speed: %{customdata[3]:.1f} km/h"
            "<extra></extra>"
        ),
    ),
    row=2, col=1
)

# Trace 5 — moving gradient point
fig_interactive.add_trace(
    go.Scatter(
        x=[plot_df.loc[0, "cumulative_distance_km"]],
        y=[plot_df.loc[0, "grade_smooth_pct"]],
        mode="markers",
        marker=dict(size=11),
        showlegend=False,
        hovertemplate="Gradient: %{y:.2f}%<extra></extra>",
    ),
    row=2, col=1
)

# Trace 6 — speed / cumulative climbing
fig_interactive.add_trace(
    go.Scatter(
        x=plot_df["cumulative_distance_km"],
        y=panel4_values,
        mode="lines",
        name=panel4_name,
        customdata=custom,
        hovertemplate=(
            "Distance: %{x:.2f} km<br>"
            + panel4_y_title + ": %{y:.2f}<br>"
            "Elevation: %{customdata[1]:.1f} m<br>"
            "Gradient: %{customdata[2]:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=2, col=2
)

# Trace 7 — moving speed/climbing point
fig_interactive.add_trace(
    go.Scatter(
        x=[plot_df.loc[0, "cumulative_distance_km"]],
        y=[panel4_values.iloc[0]],
        mode="markers",
        marker=dict(size=11),
        showlegend=False,
        hovertemplate=panel4_y_title + ": %{y:.2f}<extra></extra>",
    ),
    row=2, col=2
)

# Build animation frames that move the persistent selected point in all four panels.
# The ride-position slider can be dragged manually; Play/Pause moves it automatically.
frames = []
for idx in frame_indices:
    row = plot_df.loc[idx]
    frames.append(
        go.Frame(
            name=str(int(idx)),
            traces=[1, 3, 5, 7],
            data=[
                go.Scatter(x=[row["longitude"]], y=[row["latitude"]]),
                go.Scatter(
                    x=[row["cumulative_distance_km"]],
                    y=[row["elevation_smooth_m"]],
                ),
                go.Scatter(
                    x=[row["cumulative_distance_km"]],
                    y=[row["grade_smooth_pct"]],
                ),
                go.Scatter(
                    x=[row["cumulative_distance_km"]],
                    y=[panel4_values.iloc[int(idx)]],
                ),
            ],
        )
    )

fig_interactive.frames = frames

slider_steps = []
for idx in frame_indices:
    distance_label = plot_df.loc[int(idx), "cumulative_distance_km"]
    slider_steps.append(
        {
            "method": "animate",
            "label": f"{distance_label:.1f}",
            "args": [
                [str(int(idx))],
                {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": False},
                    "transition": {"duration": 0},
                },
            ],
        }
    )

fig_interactive.update_layout(
    height=850,
    title="Interactive Ride Dashboard",
    hovermode="closest",
    margin=dict(l=55, r=35, t=75, b=45),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "x": 0.0,
            "y": -0.08,
            "showactive": False,
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "fromcurrent": True,
                            "frame": {"duration": 80, "redraw": False},
                            "transition": {"duration": 0},
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "mode": "immediate",
                            "frame": {"duration": 0, "redraw": False},
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        },
        {
            "type": "buttons",
            "direction": "left",
            "x": 0.23,
            "y": -0.08,
            "showactive": False,
            "buttons": [
                {
                    "label": "↺ Reset zoom",
                    "method": "relayout",
                    "args": [{
                        "xaxis.autorange": True,
                        "yaxis.autorange": True,
                        "xaxis2.autorange": True,
                        "yaxis2.autorange": True,
                        "xaxis3.autorange": True,
                        "yaxis3.autorange": True,
                        "xaxis4.autorange": True,
                        "yaxis4.autorange": True,
                    }],
                }
            ],
        },
    ],
    sliders=[
        {
            "active": 0,
            "currentvalue": {"prefix": "Selected ride position — distance: "},
            "pad": {"t": 45},
            "transition": {"duration": 0},
            "steps": slider_steps,
        }
    ],
)

fig_interactive.update_xaxes(title_text="Longitude", row=1, col=1)
fig_interactive.update_yaxes(title_text="Latitude", row=1, col=1)
fig_interactive.update_xaxes(title_text="Distance (km)", row=1, col=2)
fig_interactive.update_yaxes(title_text="Elevation (m)", row=1, col=2)
fig_interactive.update_xaxes(title_text="Distance (km)", row=2, col=1)
fig_interactive.update_yaxes(title_text="Gradient (%)", row=2, col=1)
fig_interactive.update_xaxes(title_text="Distance (km)", row=2, col=2)
fig_interactive.update_yaxes(title_text=panel4_y_title, row=2, col=2)

fig_interactive.show(config={
    "displaylogo": False,
    "displayModeBar": True,
    "scrollZoom": True,
    "responsive": True,
    "modeBarButtonsToRemove": ["lasso2d", "select2d"],
})

INTERACTIVE_DASHBOARD_PATH = WEBSITE_DIR / "interactive_ride_dashboard.html"

fig_interactive.write_html(
    INTERACTIVE_DASHBOARD_PATH,
    include_plotlyjs=True,
    full_html=True,
    config={
        "displaylogo": False,
        "displayModeBar": True,
        "scrollZoom": True,
        "responsive": True,
        "modeBarButtonsToRemove": ["lasso2d", "select2d"],
    },
)

print("Interactive dashboard saved:", INTERACTIVE_DASHBOARD_PATH)



## Full interactive graph suite — one full-width graph at a time

The animated **Ride Player** above remains the overview.

Every analytical output below is now deliberately **full width**. There are no 2×2, 3×2 or other subplot arrays.

Each chart gets its own full horizontal canvas so it is easier to:

- hover exact values;
- drag and zoom;
- pan;
- reset axes;
- inspect dense sections;
- show an employer one idea at a time.

### Motion through time
Distance, displacement, speed, velocity components, acceleration, elevation, gradient and vertical speed.

### Terrain and climbing
Elevation, gradient, cumulative ascent/loss, speed over distance and terrain-performance relationships.

### Curiosity + data quality
Sampling behaviour, GPS segment spacing, speed and acceleration distributions, missingness, moving/stopped state, bearing and unusual observations.

The time-based plots are only interpreted as measured ride performance when the GPX contains genuine timestamps. If the source has no usable timestamps, the notebook retains the documented placeholder analysis clock and labels those outputs accordingly.


In [ ]:

follow_execution()

# ============================================================
# FULL-WIDTH INTERACTIVE PLOTLY GRAPH SUITE
# One graph per output — no subplot arrays.
# ============================================================

import plotly.io as pio

PLOTLY_CONFIG = {
    "displaylogo": False,
    "displayModeBar": True,
    "scrollZoom": True,
    "responsive": True,
}

TIME_PLOT_TITLE = "placeholder analysis time" if TIME_IS_SYNTHETIC else "elapsed ride time"

INTERACTIVE_MAX_POINTS = 3500
interactive_step = max(1, int(np.ceil(len(df) / INTERACTIVE_MAX_POINTS)))
idf = df.iloc[::interactive_step].copy()
if idf.index[-1] != df.index[-1]:
    idf = pd.concat([idf, df.tail(1)])
idf = idf.reset_index(drop=True)

# Display-only clipping for noisy derived series.
grade_clean_all = df["grade_smooth_pct"].replace([np.inf, -np.inf], np.nan).dropna()
if not grade_clean_all.empty:
    grade_lo, grade_hi = grade_clean_all.quantile([0.01, 0.99])
    idf["grade_display_pct"] = idf["grade_smooth_pct"].clip(grade_lo, grade_hi)
else:
    idf["grade_display_pct"] = idf["grade_smooth_pct"]

acc_valid = df["acceleration_smooth_m_s2"].replace([np.inf, -np.inf], np.nan).dropna()
if not acc_valid.empty:
    acc_lo, acc_hi = acc_valid.quantile([0.01, 0.99])
    idf["acceleration_display_m_s2"] = idf["acceleration_smooth_m_s2"].clip(acc_lo, acc_hi)
else:
    idf["acceleration_display_m_s2"] = idf["acceleration_smooth_m_s2"]

t = idf["elapsed_time_min"]
d = idf["cumulative_distance_km"]

def full_width_figure(title, x_title, y_title, height=560, hovermode="closest"):
    fig = go.Figure()
    fig.update_layout(
        title=title,
        height=height,
        hovermode=hovermode,
        margin=dict(l=65, r=35, t=78, b=60),
        autosize=True,
    )
    fig.update_xaxes(title_text=x_title, automargin=True)
    fig.update_yaxes(title_text=y_title, automargin=True)
    return fig

def show_and_save(fig, filename):
    path = WEBSITE_DIR / filename
    fig.show(config=PLOTLY_CONFIG)
    fig.write_html(
        path,
        include_plotlyjs=True,
        full_html=True,
        config=PLOTLY_CONFIG,
    )
    return path

interactive_figures = []
interactive_paths = []

# ------------------------------------------------------------
# MOTION THROUGH TIME
# ------------------------------------------------------------

fig = full_width_figure(
    f"Distance vs time — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "Cumulative distance (km)", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=t, y=idf["cumulative_distance_km"], mode="lines", name="Cumulative distance",
    hovertemplate="Time %{x:.2f} min<br>Distance %{y:.3f} km<extra></extra>"
))
interactive_figures.append(("Motion", "Distance vs time", fig))
interactive_paths.append(show_and_save(fig, "interactive_distance_time.html"))

fig = full_width_figure(
    f"Displacement from start vs time — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "Displacement from start (km)", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=t, y=idf["displacement_from_start_m"] / 1000.0, mode="lines", name="Displacement",
    hovertemplate="Time %{x:.2f} min<br>Displacement %{y:.3f} km<extra></extra>"
))
interactive_figures.append(("Motion", "Displacement vs time", fig))
interactive_paths.append(show_and_save(fig, "interactive_displacement_time.html"))

fig = full_width_figure(
    f"Speed vs time — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "Speed (km/h)", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=t, y=idf["speed_kmh_raw"], mode="lines", name="Raw speed", opacity=0.28,
    hovertemplate="Time %{x:.2f} min<br>Raw speed %{y:.2f} km/h<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=t, y=idf["speed_smooth_kmh"], mode="lines", name="Smoothed speed",
    hovertemplate="Time %{x:.2f} min<br>Smoothed speed %{y:.2f} km/h<extra></extra>"
))
interactive_figures.append(("Motion", "Speed vs time", fig))
interactive_paths.append(show_and_save(fig, "interactive_speed_time.html"))

fig = full_width_figure(
    f"Velocity components vs time — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "Velocity (m/s)", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=t, y=idf["velocity_east_m_s"], mode="lines", name="East velocity",
    hovertemplate="Time %{x:.2f} min<br>East velocity %{y:.3f} m/s<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=t, y=idf["velocity_north_m_s"], mode="lines", name="North velocity",
    hovertemplate="Time %{x:.2f} min<br>North velocity %{y:.3f} m/s<extra></extra>"
))
interactive_figures.append(("Motion", "Velocity components vs time", fig))
interactive_paths.append(show_and_save(fig, "interactive_velocity_time.html"))

fig = full_width_figure(
    f"Tangential acceleration vs time — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "Acceleration (m/s²)", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=t, y=idf["acceleration_display_m_s2"], mode="lines", name="Acceleration",
    hovertemplate="Time %{x:.2f} min<br>Acceleration %{y:.3f} m/s²<extra></extra>"
))
interactive_figures.append(("Motion", "Acceleration vs time", fig))
interactive_paths.append(show_and_save(fig, "interactive_acceleration_time.html"))

fig = full_width_figure(
    f"Elevation vs time — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "Elevation (m)", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=t, y=idf["elevation_smooth_m"], mode="lines", name="Elevation",
    hovertemplate="Time %{x:.2f} min<br>Elevation %{y:.1f} m<extra></extra>"
))
interactive_figures.append(("Motion", "Elevation vs time", fig))
interactive_paths.append(show_and_save(fig, "interactive_elevation_time.html"))

fig = full_width_figure(
    f"Gradient vs time — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "Gradient (%)", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=t, y=idf["grade_display_pct"], mode="lines", name="Gradient",
    hovertemplate="Time %{x:.2f} min<br>Gradient %{y:.2f}%<extra></extra>"
))
interactive_figures.append(("Motion", "Gradient vs time", fig))
interactive_paths.append(show_and_save(fig, "interactive_gradient_time.html"))

fig = full_width_figure(
    f"Vertical speed vs time — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "Vertical speed (m/s)", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=t, y=idf["vertical_speed_m_s"], mode="lines", name="Vertical speed",
    hovertemplate="Time %{x:.2f} min<br>Vertical speed %{y:.3f} m/s<extra></extra>"
))
interactive_figures.append(("Motion", "Vertical speed vs time", fig))
interactive_paths.append(show_and_save(fig, "interactive_vertical_speed_time.html"))

# ------------------------------------------------------------
# TERRAIN + CLIMBING
# ------------------------------------------------------------

fig = full_width_figure("Elevation profile", "Distance (km)", "Elevation (m)", hovermode="x unified")
fig.add_trace(go.Scatter(
    x=d, y=idf["elevation_m"], mode="lines", name="Recorded elevation", opacity=0.28,
    hovertemplate="Distance %{x:.3f} km<br>Recorded elevation %{y:.1f} m<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=d, y=idf["elevation_smooth_m"], mode="lines", name="Smoothed elevation",
    hovertemplate="Distance %{x:.3f} km<br>Elevation %{y:.1f} m<extra></extra>"
))
interactive_figures.append(("Terrain", "Elevation profile", fig))
interactive_paths.append(show_and_save(fig, "interactive_elevation_profile.html"))

fig = full_width_figure("Gradient vs distance", "Distance (km)", "Gradient (%)", hovermode="x unified")
fig.add_trace(go.Scatter(
    x=d, y=idf["grade_display_pct"], mode="lines", name="Gradient",
    hovertemplate="Distance %{x:.3f} km<br>Gradient %{y:.2f}%<extra></extra>"
))
interactive_figures.append(("Terrain", "Gradient vs distance", fig))
interactive_paths.append(show_and_save(fig, "interactive_gradient_distance.html"))

fig = full_width_figure("Cumulative ascent and descent", "Distance (km)", "Vertical metres", hovermode="x unified")
fig.add_trace(go.Scatter(
    x=d, y=idf["cumulative_elevation_gain_m"], mode="lines", name="Cumulative ascent",
    hovertemplate="Distance %{x:.3f} km<br>Ascent %{y:.1f} m<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=d, y=idf["cumulative_elevation_loss_m"], mode="lines", name="Cumulative descent",
    hovertemplate="Distance %{x:.3f} km<br>Descent %{y:.1f} m<extra></extra>"
))
interactive_figures.append(("Terrain", "Cumulative ascent and descent", fig))
interactive_paths.append(show_and_save(fig, "interactive_ascent_descent.html"))

fig = full_width_figure("Speed vs distance", "Distance (km)", "Speed (km/h)", hovermode="x unified")
fig.add_trace(go.Scatter(
    x=d, y=idf["speed_smooth_kmh"], mode="lines", name="Speed",
    hovertemplate="Distance %{x:.3f} km<br>Speed %{y:.2f} km/h<extra></extra>"
))
interactive_figures.append(("Terrain", "Speed vs distance", fig))
interactive_paths.append(show_and_save(fig, "interactive_speed_distance.html"))

scatter_rel = idf[
    ["grade_smooth_pct", "speed_smooth_kmh", "elevation_smooth_m", "cumulative_distance_km"]
].replace([np.inf, -np.inf], np.nan).dropna()

if not scatter_rel.empty:
    sg1, sg2 = scatter_rel["grade_smooth_pct"].quantile([0.01, 0.99])
    ss1, ss2 = scatter_rel["speed_smooth_kmh"].quantile([0.01, 0.99])
    scatter_rel = scatter_rel[
        scatter_rel["grade_smooth_pct"].between(sg1, sg2)
        & scatter_rel["speed_smooth_kmh"].between(ss1, ss2)
    ]

fig = full_width_figure("Speed vs gradient", "Gradient (%)", "Speed (km/h)")
fig.add_trace(go.Scatter(
    x=scatter_rel["grade_smooth_pct"],
    y=scatter_rel["speed_smooth_kmh"],
    mode="markers",
    name="Ride points",
    marker=dict(size=6, opacity=0.40),
    customdata=np.column_stack([
        scatter_rel["cumulative_distance_km"],
        scatter_rel["elevation_smooth_m"],
    ]) if len(scatter_rel) else None,
    hovertemplate=(
        "Gradient %{x:.2f}%<br>Speed %{y:.2f} km/h<br>"
        "Distance %{customdata[0]:.2f} km<br>Elevation %{customdata[1]:.1f} m"
        "<extra></extra>"
    ),
))
interactive_figures.append(("Terrain", "Speed vs gradient", fig))
interactive_paths.append(show_and_save(fig, "interactive_speed_gradient.html"))

fig = full_width_figure("Speed vs elevation", "Elevation (m)", "Speed (km/h)")
fig.add_trace(go.Scatter(
    x=scatter_rel["elevation_smooth_m"],
    y=scatter_rel["speed_smooth_kmh"],
    mode="markers",
    name="Ride points",
    marker=dict(size=6, opacity=0.40),
    customdata=np.column_stack([
        scatter_rel["cumulative_distance_km"],
        scatter_rel["grade_smooth_pct"],
    ]) if len(scatter_rel) else None,
    hovertemplate=(
        "Elevation %{x:.1f} m<br>Speed %{y:.2f} km/h<br>"
        "Distance %{customdata[0]:.2f} km<br>Gradient %{customdata[1]:.2f}%"
        "<extra></extra>"
    ),
))
interactive_figures.append(("Terrain", "Speed vs elevation", fig))
interactive_paths.append(show_and_save(fig, "interactive_speed_elevation.html"))

# ------------------------------------------------------------
# CURIOSITY + DATA QUALITY
# ------------------------------------------------------------

sampling_plot = df["delta_time_s"].replace([np.inf, -np.inf], np.nan).dropna()
if not sampling_plot.empty:
    sampling_plot = sampling_plot.clip(upper=sampling_plot.quantile(0.99))
fig = full_width_figure("Sampling interval distribution", "Sampling interval (s)", "Count")
fig.add_trace(go.Histogram(
    x=sampling_plot, nbinsx=50, name="Sampling interval",
    hovertemplate="Interval %{x:.2f} s<br>Count %{y}<extra></extra>"
))
interactive_figures.append(("Quality", "Sampling interval distribution", fig))
interactive_paths.append(show_and_save(fig, "interactive_sampling_distribution.html"))

segment_plot = df["segment_distance_m"].replace([np.inf, -np.inf], np.nan).dropna()
if not segment_plot.empty:
    segment_plot = segment_plot.clip(upper=segment_plot.quantile(0.99))
fig = full_width_figure("GPS segment-distance distribution", "Segment distance (m)", "Count")
fig.add_trace(go.Histogram(
    x=segment_plot, nbinsx=50, name="Segment distance",
    hovertemplate="Segment %{x:.2f} m<br>Count %{y}<extra></extra>"
))
interactive_figures.append(("Quality", "GPS segment-distance distribution", fig))
interactive_paths.append(show_and_save(fig, "interactive_segment_distribution.html"))

speed_dist = df.loc[~df["speed_outlier_flag"], "speed_smooth_kmh"].replace(
    [np.inf, -np.inf], np.nan
).dropna()
fig = full_width_figure("Speed distribution", "Speed (km/h)", "Count")
fig.add_trace(go.Histogram(
    x=speed_dist, nbinsx=50, name="Speed distribution",
    hovertemplate="Speed %{x:.2f} km/h<br>Count %{y}<extra></extra>"
))
interactive_figures.append(("Quality", "Speed distribution", fig))
interactive_paths.append(show_and_save(fig, "interactive_speed_distribution.html"))

acc_dist = df["acceleration_smooth_m_s2"].replace([np.inf, -np.inf], np.nan).dropna()
if not acc_dist.empty:
    a1, a2 = acc_dist.quantile([0.01, 0.99])
    acc_dist = acc_dist.clip(a1, a2)
fig = full_width_figure("Acceleration distribution", "Acceleration (m/s²)", "Count")
fig.add_trace(go.Histogram(
    x=acc_dist, nbinsx=50, name="Acceleration distribution",
    hovertemplate="Acceleration %{x:.3f} m/s²<br>Count %{y}<extra></extra>"
))
interactive_figures.append(("Quality", "Acceleration distribution", fig))
interactive_paths.append(show_and_save(fig, "interactive_acceleration_distribution.html"))

raw_missing = raw_df.isna().sum()
fig = full_width_figure("Raw-source missing values", "Raw field", "Missing values")
fig.add_trace(go.Bar(
    x=raw_missing.index.astype(str), y=raw_missing.values, name="Missing values",
    hovertemplate="%{x}<br>Missing %{y}<extra></extra>"
))
interactive_figures.append(("Quality", "Raw-source missing values", fig))
interactive_paths.append(show_and_save(fig, "interactive_missingness.html"))

fig = full_width_figure(
    f"Moving / stopped state — {TIME_PLOT_TITLE}",
    "Elapsed time (min)", "0 = stopped, 1 = moving", hovermode="x unified"
)
fig.add_trace(go.Scatter(
    x=idf["elapsed_time_min"], y=idf["is_moving"].astype(float),
    mode="lines", line_shape="hv", name="Moving state",
    hovertemplate="Time %{x:.2f} min<br>Moving state %{y:.0f}<extra></extra>"
))
interactive_figures.append(("Quality", "Moving / stopped state", fig))
interactive_paths.append(show_and_save(fig, "interactive_moving_state.html"))

fig = full_width_figure("Bearing through the route", "Distance (km)", "Bearing (degrees)")
fig.add_trace(go.Scatter(
    x=idf["cumulative_distance_km"], y=idf["bearing_deg"],
    mode="markers", marker=dict(size=5, opacity=0.45), name="Bearing",
    hovertemplate="Distance %{x:.2f} km<br>Bearing %{y:.1f}°<extra></extra>"
))
interactive_figures.append(("Quality", "Bearing through the route", fig))
interactive_paths.append(show_and_save(fig, "interactive_bearing.html"))

flagged = df.loc[
    df["speed_outlier_flag"].fillna(False),
    ["elapsed_time_min", "speed_kmh_raw", "segment_distance_m", "delta_time_s"]
].copy()
fig = full_width_figure(
    "Flagged speed observations",
    "Elapsed time (min)", "Raw speed (km/h)"
)
fig.add_trace(go.Scatter(
    x=flagged["elapsed_time_min"], y=flagged["speed_kmh_raw"],
    mode="markers", marker=dict(size=8), name="Flagged observations",
    customdata=np.column_stack([
        flagged["segment_distance_m"],
        flagged["delta_time_s"],
    ]) if len(flagged) else None,
    hovertemplate=(
        "Time %{x:.2f} min<br>Raw speed %{y:.2f} km/h<br>"
        "Segment %{customdata[0]:.1f} m<br>Δt %{customdata[1]:.2f} s"
        "<extra></extra>"
    ),
))
interactive_figures.append(("Quality", "Flagged speed observations", fig))
interactive_paths.append(show_and_save(fig, "interactive_flagged_speed.html"))

# ------------------------------------------------------------
# CURIOUS / INTERVIEW-WORTHY FINDINGS
# ------------------------------------------------------------

curiosity_rows = []

def add_finding(metric, value, note):
    curiosity_rows.append({"metric": metric, "value": value, "note": note})

farthest_idx = int(df["displacement_from_start_m"].idxmax())
add_finding(
    "Farthest point from start",
    f"{df.loc[farthest_idx, 'displacement_from_start_m'] / 1000.0:.2f} km",
    f"At {df.loc[farthest_idx, 'cumulative_distance_km']:.2f} km travelled."
)

highest_idx = int(df["elevation_smooth_m"].idxmax())
add_finding(
    "Highest smoothed elevation",
    f"{df.loc[highest_idx, 'elevation_smooth_m']:.1f} m",
    f"At {df.loc[highest_idx, 'cumulative_distance_km']:.2f} km."
)

if not grade_clean_all.empty:
    g1, g99 = grade_clean_all.quantile([0.01, 0.99])
    robust_grade = grade_clean_all.clip(g1, g99)
    steep_idx = int(robust_grade.idxmax())
    add_finding(
        "Steepest robust gradient",
        f"{robust_grade.loc[steep_idx]:.1f}%",
        "Smoothed and screened at the 1st–99th percentiles for display robustness."
    )

speed_clean_find = df["speed_smooth_kmh"].replace([np.inf, -np.inf], np.nan).dropna()
if not speed_clean_find.empty:
    fast_idx = int(speed_clean_find.idxmax())
    add_finding(
        "Highest smoothed speed",
        f"{speed_clean_find.loc[fast_idx]:.1f} km/h",
        f"At {df.loc[fast_idx, 'cumulative_distance_km']:.2f} km."
    )

acc_find = df["acceleration_smooth_m_s2"].replace([np.inf, -np.inf], np.nan).dropna()
if not acc_find.empty:
    add_finding(
        "Strongest smoothed acceleration",
        f"{acc_find.max():.3f} m/s²",
        "Approximate tangential acceleration from change in smoothed speed."
    )
    add_finding(
        "Strongest smoothed deceleration",
        f"{acc_find.min():.3f} m/s²",
        "Negative values indicate reduction in speed."
    )

corr_source = df[
    ["speed_smooth_kmh", "grade_smooth_pct", "elevation_smooth_m", "vertical_speed_m_s"]
].replace([np.inf, -np.inf], np.nan).dropna()

if len(corr_source) >= 3:
    add_finding(
        "Speed–gradient correlation",
        f"{corr_source['speed_smooth_kmh'].corr(corr_source['grade_smooth_pct']):.3f}",
        "Descriptive association only; it does not establish causation."
    )
    add_finding(
        "Speed–elevation correlation",
        f"{corr_source['speed_smooth_kmh'].corr(corr_source['elevation_smooth_m']):.3f}",
        "Descriptive association only."
    )

curiosity_findings = pd.DataFrame(curiosity_rows)
display(curiosity_findings)
curiosity_findings.to_csv(DATA_DIR / "curiosity_findings.csv", index=False)

# ------------------------------------------------------------
# ALL-IN-ONE EMPLOYER PAGE — EVERY OUTPUT FULL WIDTH
# ------------------------------------------------------------

FULL_INTERACTIVE_PATH = WEBSITE_DIR / "interactive_full_analysis.html"

time_warning = (
    "<strong>Time provenance:</strong> genuine GPX timestamps are being used."
    if HAS_REAL_TIME
    else
    "<strong>Time provenance:</strong> the source GPX lacked a complete genuine timestamp series. "
    "Time-dependent outputs therefore use the notebook's clearly labelled placeholder analysis clock "
    "and must not be presented as recovered Strava performance."
)

sections = []
for group, title, fig in interactive_figures:
    sections.append(
        f"""
        <section class="chart-section" data-group="{group}">
          <h2>{title}</h2>
          <div class="plot-wrap">
            {pio.to_html(fig, include_plotlyjs=False, full_html=False, config=PLOTLY_CONFIG)}
          </div>
        </section>
        """
    )

full_html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{ACTIVITY_TITLE} — Full Interactive Analysis</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
*{{box-sizing:border-box}}
body{{font-family:Inter,system-ui,-apple-system,"Segoe UI",sans-serif;margin:0;color:#0f172a;background:#fff;line-height:1.5}}
main{{width:100%;margin:0;padding:24px 18px 64px}}
.header-inner{{width:min(1500px,100%);margin:auto}}
h1{{font-size:clamp(2rem,4vw,3.4rem);margin:.2rem 0}}
h2{{margin:0 0 10px;font-size:clamp(1.15rem,2vw,1.55rem)}}
p{{max-width:1050px;color:#475569}}
nav{{display:flex;gap:8px;flex-wrap:wrap;margin:18px 0 28px}}
nav a{{text-decoration:none;color:inherit;border:1px solid #dbe2ea;border-radius:999px;padding:8px 12px}}
.note{{border:1px solid #dbe2ea;background:#f8fafc;border-radius:12px;padding:12px 14px;margin:14px 0}}
.chart-section{{width:100%;margin:0 0 34px;padding:0}}
.plot-wrap{{width:100%;border:1px solid #dbe2ea;border-radius:14px;padding:8px;overflow:hidden;background:#fff}}
.plot-wrap>div{{width:100%!important}}
.js-plotly-plot,.plot-container,.svg-container{{width:100%!important}}
.small{{font-size:.86rem;color:#64748b}}
@media(max-width:700px){{main{{padding:18px 8px 46px}}.plot-wrap{{padding:2px}}}}
</style>
</head>
<body>
<main>
<div class="header-inner">
  <div class="small">GPX · PANDAS · KINEMATICS · TERRAIN · DATA QUALITY · PLOTLY</div>
  <h1>{ACTIVITY_TITLE} — Full Interactive Analysis</h1>
  <p>Every analytical output is full width. Hover for exact values, zoom and pan, reset axes, and export chart images from the Plotly toolbar.</p>
  <div class="note">{time_warning}</div>
  <nav>
    <a href="#motion">Motion through time</a>
    <a href="#terrain">Terrain and climbing</a>
    <a href="#quality">Curiosity + data quality</a>
  </nav>
</div>

<div id="motion"></div>
{''.join(s for s in sections if 'data-group="Motion"' in s)}
<div id="terrain"></div>
{''.join(s for s in sections if 'data-group="Terrain"' in s)}
<div id="quality"></div>
{''.join(s for s in sections if 'data-group="Quality"' in s)}
</main>
</body>
</html>"""

FULL_INTERACTIVE_PATH.write_text(full_html, encoding="utf-8")

interactive_catalog = pd.DataFrame(
    [
        [group, title, path.name]
        for (group, title, _), path in zip(interactive_figures, interactive_paths)
    ]
    + [["All", "Full interactive analysis", FULL_INTERACTIVE_PATH.name]],
    columns=["group", "interactive_output", "file"]
)

display(interactive_catalog)
interactive_catalog.to_csv(DATA_DIR / "interactive_output_catalog.csv", index=False)

print("Full-width interactive graph suite saved.")
print(f"Individual full-width charts: {len(interactive_paths)}")
print("All-in-one full-width page:", FULL_INTERACTIVE_PATH)



## Part E — Plotting the data with Matplotlib

The interactive map has already been saved above. The remaining figures are the plotting lesson.

Each chart is:

1. displayed directly in Google Colab;
2. exported as a high-resolution PNG;
3. exported as SVG for crisp website use;
4. registered in `chart_catalog.json` with an explanation of what it shows.

### Plotting checkpoint

For every graph, identify four things in the code:

**figure / axes → x variable → y variable → labels**

Then ask the analytical question the graph answers. For example:

- line plot: **how does a quantity change through the ride?**
- histogram: **what is the distribution?**
- scatter plot: **are two variables associated?**
- two y-axes: **how do two differently scaled variables move together?**

If placeholder time is being used, the notebook still demonstrates the plotting code, but time-derived charts are explicitly labelled as placeholder-time examples.


In [ ]:
follow_execution()

def finish_plot(name):
    plt.tight_layout()
    stem = Path(name).stem
    png = FIGURES_DIR / f"{stem}.png"
    svg = FIGURES_DIR / f"{stem}.svg"
    plt.savefig(png, dpi=200, bbox_inches="tight")
    plt.savefig(svg, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", png)
    print("Saved:", svg)


def unavailable_plot(name, title, reason):
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.axis("off")
    ax.text(
        0.5, 0.58, title,
        ha="center", va="center",
        fontsize=18, weight="bold",
        transform=ax.transAxes
    )
    ax.text(
        0.5, 0.40, reason,
        ha="center", va="center",
        fontsize=12,
        wrap=True,
        transform=ax.transAxes
    )
    finish_plot(name)

print("Primary visual overview already shown above: full interactive route map.")
print("The following figures are the analytical figure set exported for the website.")

# 02 — Sampling interval
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(sampling.clip(upper=sampling.quantile(0.99)), bins=40)
    ax.axvline(sampling.median(), linestyle="--", label="Median")
    ax.set_title("GPX sampling interval")
    ax.set_xlabel("Seconds between recorded points")
    ax.set_ylabel("Track points")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    finish_plot("02_sampling_interval")
else:
    unavailable_plot(
        "02_sampling_interval",
        "Sampling interval unavailable",
        "No complete timestamp series exists in this GPX."
    )

# 03 — Geography is intentionally represented by the single interactive map above.
# No second route-shape figure is generated.

# 04 — Cumulative distance versus time
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["elapsed_time_min"], df["cumulative_distance_km"], linewidth=2)
    ax.set_title("Cumulative distance versus " + ("placeholder time" if TIME_IS_SYNTHETIC else "elapsed time"))
    ax.set_xlabel(TIME_AXIS_LABEL)
    ax.set_ylabel("Cumulative distance (km)")
    ax.grid(alpha=0.25)
    finish_plot("04_distance_vs_time")
else:
    unavailable_plot(
        "04_distance_vs_time",
        "Distance vs time unavailable",
        "Cumulative distance is available, but a real time axis requires genuine GPX timestamps."
    )

# 05 — Displacement magnitude versus time
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["elapsed_time_min"], df["displacement_from_start_m"] / 1000.0, linewidth=2)
    ax.set_title("Displacement magnitude versus " + ("placeholder time" if TIME_IS_SYNTHETIC else "elapsed time"))
    ax.set_xlabel(TIME_AXIS_LABEL)
    ax.set_ylabel("Displacement from start (km)")
    ax.grid(alpha=0.25)
    finish_plot("05_displacement_vs_time")
else:
    unavailable_plot(
        "05_displacement_vs_time",
        "Displacement vs time unavailable",
        "Displacement from the start exists, but this source has no complete time axis."
    )

# 06 — Elevation vs distance
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["cumulative_distance_km"], df["elevation_m"], linewidth=0.8, alpha=0.3, label="Recorded")
ax.plot(df["cumulative_distance_km"], df["elevation_smooth_m"], linewidth=2, label="Smoothed")
ax.set_title("Elevation profile")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Elevation (m)")
ax.legend()
ax.grid(alpha=0.25)
finish_plot("06_elevation_vs_distance")

# 07 — Elevation vs time
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["elapsed_time_min"], df["elevation_smooth_m"], linewidth=2)
    ax.set_title("Elevation versus " + ("placeholder time" if TIME_IS_SYNTHETIC else "elapsed time"))
    ax.set_xlabel(TIME_AXIS_LABEL)
    ax.set_ylabel("Elevation (m)")
    ax.grid(alpha=0.25)
    finish_plot("07_elevation_vs_time")
else:
    unavailable_plot("07_elevation_vs_time", "Elevation vs time unavailable", "Elevation exists, but the GPX has no complete timestamp series.")

# 08 — Speed vs time
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["elapsed_time_min"], df["speed_kmh_raw"], linewidth=0.7, alpha=0.25, label="Raw segment speed")
    ax.plot(df["elapsed_time_min"], df["speed_smooth_kmh"], linewidth=2, label="Smoothed speed")
    ax.set_title(("Placeholder-time " if TIME_IS_SYNTHETIC else "") + "speed versus time")
    ax.set_xlabel(TIME_AXIS_LABEL)
    ax.set_ylabel("Speed (km/h)")
    ax.legend()
    ax.grid(alpha=0.25)
    finish_plot("08_speed_vs_time")
else:
    unavailable_plot("08_speed_vs_time", "Speed vs time unavailable", "Speed requires distance divided by genuine elapsed time; time is missing from this GPX.")

# 09 — Velocity components
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["elapsed_time_min"], df["velocity_east_m_s"], linewidth=1.3, label="East velocity component")
    ax.plot(df["elapsed_time_min"], df["velocity_north_m_s"], linewidth=1.3, label="North velocity component")
    ax.axhline(0, linewidth=1)
    ax.set_title(("Placeholder-time " if TIME_IS_SYNTHETIC else "") + "velocity components versus time")
    ax.set_xlabel(TIME_AXIS_LABEL)
    ax.set_ylabel("Velocity component (m/s)")
    ax.legend()
    ax.grid(alpha=0.25)
    finish_plot("09_velocity_components_vs_time")
else:
    unavailable_plot("09_velocity_components_vs_time", "Velocity components unavailable", "Velocity components require both direction and genuine time-based speed.")

# 10 — Acceleration vs time
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["elapsed_time_min"], df["acceleration_smooth_m_s2"], linewidth=1.5)
    ax.axhline(0, linewidth=1)
    ax.set_title(("Placeholder-time " if TIME_IS_SYNTHETIC else "Approximate ") + "tangential acceleration versus time")
    ax.set_xlabel(TIME_AXIS_LABEL)
    ax.set_ylabel("Acceleration (m/s²)")
    ax.grid(alpha=0.25)
    finish_plot("10_acceleration_vs_time")
else:
    unavailable_plot("10_acceleration_vs_time", "Acceleration vs time unavailable", "Acceleration requires change in velocity over genuine elapsed time.")

# 11 — Gradient vs distance
grade_plot = df["grade_smooth_pct"].copy()
glo, ghi = grade_plot.quantile([0.01, 0.99])
grade_plot = grade_plot.clip(glo, ghi)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["cumulative_distance_km"], grade_plot, linewidth=1.5)
ax.axhline(0, linewidth=1)
ax.set_title("Smoothed gradient versus distance")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Gradient (%)")
ax.grid(alpha=0.25)
finish_plot("11_gradient_vs_distance")

# 12 — Speed and gradient on same time axis
if HAS_TIME:
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(df["elapsed_time_min"], df["speed_smooth_kmh"], linewidth=2, label="Speed")
    ax1.set_xlabel(TIME_AXIS_LABEL)
    ax1.set_ylabel("Speed (km/h)")
    ax1.grid(alpha=0.2)
    ax2 = ax1.twinx()
    ax2.plot(df["elapsed_time_min"], grade_plot, linewidth=1.2, alpha=0.7, linestyle="--", label="Gradient")
    ax2.set_ylabel("Gradient (%)")
    ax1.set_title(("Placeholder-time " if TIME_IS_SYNTHETIC else "") + "speed and gradient over the same time axis")
    l1, n1 = ax1.get_legend_handles_labels()
    l2, n2 = ax2.get_legend_handles_labels()
    ax1.legend(l1 + l2, n1 + n2, loc="upper right")
    finish_plot("12_speed_and_gradient_vs_time")
else:
    unavailable_plot("12_speed_and_gradient_vs_time", "Speed-gradient time comparison unavailable", "Gradient exists, but speed/time requires timestamps.")

# 13 — Speed vs gradient scatter
if HAS_TIME:
    scatter_df = df[["speed_smooth_kmh", "grade_smooth_pct"]].replace([np.inf, -np.inf], np.nan).dropna()
    g1, g2 = scatter_df["grade_smooth_pct"].quantile([0.01, 0.99])
    s1, s2 = scatter_df["speed_smooth_kmh"].quantile([0.01, 0.99])
    scatter_df = scatter_df[
        scatter_df["grade_smooth_pct"].between(g1, g2)
        & scatter_df["speed_smooth_kmh"].between(s1, s2)
    ]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(scatter_df["grade_smooth_pct"], scatter_df["speed_smooth_kmh"], s=10, alpha=0.35)
    ax.set_title("Speed versus gradient")
    ax.set_xlabel("Gradient (%)")
    ax.set_ylabel("Speed (km/h)")
    ax.grid(alpha=0.2)
    finish_plot("13_speed_vs_gradient_scatter")
else:
    unavailable_plot("13_speed_vs_gradient_scatter", "Speed vs gradient unavailable", "Gradient is available; speed is not because timestamps are absent.")

# 14 — Speed distribution
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(9, 5))
    sd = df.loc[~df["speed_outlier_flag"], "speed_smooth_kmh"].dropna()
    ax.hist(sd, bins=40)
    ax.axvline(sd.mean(), linestyle="--", label=f"Mean = {sd.mean():.1f} km/h")
    ax.axvline(sd.median(), linestyle=":", label=f"Median = {sd.median():.1f} km/h")
    ax.set_title("Distribution of cycling speed")
    ax.set_xlabel("Speed (km/h)")
    ax.set_ylabel("Track points")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    finish_plot("14_speed_distribution")
else:
    unavailable_plot("14_speed_distribution", "Speed distribution unavailable", "No genuine time axis is available.")

# 15 — Cumulative climb
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["cumulative_distance_km"], df["cumulative_elevation_gain_m"], linewidth=2)
ax.set_title("Cumulative elevation gain")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Elevation gain accumulated (m)")
ax.grid(alpha=0.25)
finish_plot("15_cumulative_elevation_gain")

# 16 — Elevation + gradient together
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(df["cumulative_distance_km"], df["elevation_smooth_m"], linewidth=2, label="Elevation")
ax1.set_xlabel("Distance (km)")
ax1.set_ylabel("Elevation (m)")
ax2 = ax1.twinx()
ax2.plot(df["cumulative_distance_km"], grade_plot, linewidth=1, alpha=0.65, linestyle="--", label="Gradient")
ax2.set_ylabel("Gradient (%)")
ax1.set_title("Elevation and gradient over distance")
l1, n1 = ax1.get_legend_handles_labels(); l2, n2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, n1 + n2, loc="upper right")
finish_plot("16_elevation_and_gradient")

# 17 — Segment-distance distribution
fig, ax = plt.subplots(figsize=(9, 5))
seg = df["segment_distance_m"].dropna()
ax.hist(seg.clip(upper=seg.quantile(0.99)), bins=40)
ax.axvline(seg.median(), linestyle="--", label=f"Median = {seg.median():.1f} m")
ax.set_title("Distribution of point-to-point GPS distances")
ax.set_xlabel("Segment distance (m)")
ax.set_ylabel("Segments")
ax.legend()
ax.grid(axis="y", alpha=0.25)
finish_plot("17_segment_distance_distribution")

# 18 — Acceleration distribution
if HAS_TIME:
    fig, ax = plt.subplots(figsize=(9, 5))
    acc = df["acceleration_smooth_m_s2"].dropna()
    ax.hist(acc, bins=40)
    ax.set_title("Distribution of approximate acceleration")
    ax.set_xlabel("Acceleration (m/s²)")
    ax.set_ylabel("Track points")
    ax.grid(axis="y", alpha=0.25)
    finish_plot("18_acceleration_distribution")
else:
    unavailable_plot("18_acceleration_distribution", "Acceleration distribution unavailable", "No complete timestamp series exists.")

# 19 — Raw-source missing-value audit
fig, ax = plt.subplots(figsize=(10, 4.8))
mc = raw_df.isna().sum()
ax.bar(mc.index.astype(str), mc.values)
ax.set_title("Missing values in the raw GPX data")
ax.set_ylabel("Missing values")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
finish_plot("19_raw_source_missing_values")


print("Visualisation export complete.")


## Part F — Aggregation + SQL

The raw ride may contain thousands of points. Aggregation turns those observations into kilometre-level and, when time exists, minute-level summaries. SQLite then stores both raw and engineered data so the project is not limited to an in-memory DataFrame.


### pandas checkpoint — `groupby` and `agg`

The kilometre and minute summaries are where row-level observations become management-style summaries.

`groupby()` answers **which groups?**  
`agg()` answers **which summary for each group?**

The SQLite step then shows the same analytical data persisted outside the notebook and queried with SQL.


In [ ]:
follow_execution()

df["kilometre_bin"] = np.floor(df["cumulative_distance_km"]).astype(int)

km_agg = {
    "distance_km": ("analysis_segment_distance_m", lambda s: s.sum() / 1000.0),
    "elevation_gain_m": ("elevation_gain_m", "sum"),
    "elevation_loss_m": ("elevation_loss_m", "sum"),
    "average_elevation_m": ("elevation_smooth_m", "mean"),
    "max_elevation_m": ("elevation_smooth_m", "max"),
    "average_grade_pct": ("grade_smooth_pct", "mean"),
}
if HAS_TIME:
    km_agg.update({
        "average_speed_kmh": ("speed_smooth_kmh", "mean"),
        "max_speed_kmh": ("speed_smooth_kmh", "max"),
    })

km_summary = df.groupby("kilometre_bin", as_index=False).agg(**km_agg)
display(km_summary.head(30))
km_summary.to_csv(DATA_DIR / "kilometre_summary.csv", index=False)

if HAS_TIME:
    df["minute_bin"] = np.floor(df["elapsed_time_min"]).astype(int)
    minute_summary = (
        df.groupby("minute_bin", as_index=False)
        .agg(
            distance_km=("analysis_segment_distance_m", lambda s: s.sum() / 1000.0),
            average_speed_kmh=("speed_smooth_kmh", "mean"),
            max_speed_kmh=("speed_smooth_kmh", "max"),
            average_elevation_m=("elevation_smooth_m", "mean"),
            average_grade_pct=("grade_smooth_pct", "mean"),
        )
    )
else:
    minute_summary = pd.DataFrame()

if not minute_summary.empty:
    display(minute_summary.head(30))
    minute_summary.to_csv(DATA_DIR / "minute_summary.csv", index=False)

DB_PATH = SQL_DIR / "ride_analysis.db"
with sqlite3.connect(DB_PATH) as conn:
    raw_df.to_sql("raw_track_points", conn, if_exists="replace", index=False)
    df.to_sql("engineered_track_points", conn, if_exists="replace", index=False)
    ride_summary.to_sql("ride_summary", conn, if_exists="replace", index=False)
    data_quality.to_sql("data_quality", conn, if_exists="replace", index=False)
    km_summary.to_sql("kilometre_summary", conn, if_exists="replace", index=False)
    if not minute_summary.empty:
        minute_summary.to_sql("minute_summary", conn, if_exists="replace", index=False)

queries = OrderedDict()
queries["ride_summary"] = "SELECT * FROM ride_summary"
queries["highest_points"] = """
SELECT point_id, cumulative_distance_km, elevation_smooth_m, grade_smooth_pct
FROM engineered_track_points
ORDER BY elevation_smooth_m DESC
LIMIT 10
"""
queries["most_climbing_km"] = """
SELECT *
FROM kilometre_summary
ORDER BY elevation_gain_m DESC
LIMIT 10
"""
queries["largest_segments"] = """
SELECT point_id, segment_distance_m, cumulative_distance_km
FROM engineered_track_points
ORDER BY segment_distance_m DESC
LIMIT 10
"""
if HAS_TIME:
    queries["fastest_km"] = """
SELECT kilometre_bin, average_speed_kmh, max_speed_kmh, average_grade_pct
FROM kilometre_summary
ORDER BY average_speed_kmh DESC
LIMIT 10
"""
    queries["speed_flags"] = """
SELECT point_id, elapsed_time_min, segment_distance_m, delta_time_s, speed_kmh_raw
FROM engineered_track_points
WHERE speed_outlier_flag = 1
ORDER BY speed_kmh_raw DESC
LIMIT 20
"""

query_results = {}
with sqlite3.connect(DB_PATH) as conn:
    for title, query in queries.items():
        print("\nSQL:", title)
        result = pd.read_sql_query(query, conn)
        display(result)
        query_results[title] = result

(SQL_DIR / "example_queries.sql").write_text(
    "\n\n".join(
        f"-- {title}\n{query.strip()};"
        for title, query in queries.items()
    ),
    encoding="utf-8"
)

print("SQLite database:", DB_PATH)


## Part H — Export a Complete Web Dataset

The later website should not have to repeat the Python analysis.

This export includes not just numbers, but enough context to **teach the project**:

- source metadata;
- data dictionary;
- quality checks;
- descriptive statistics;
- chart catalogue;
- ride narrative;
- critical code snippets + explanations;
- SQL examples + results;
- full/downsampled time series;
- GeoJSON route;
- website-ready JavaScript data.


In [ ]:
follow_execution()

shutil.copy2(GPX_PATH, SOURCE_DIR / GPX_PATH.name)

# Core tables
raw_df.to_csv(DATA_DIR / "raw_track_points.csv", index=False)
df.to_csv(DATA_DIR / "engineered_track_points.csv", index=False)
missing_table.to_csv(DATA_DIR / "missing_values.csv", index=False)
dtype_table.to_csv(DATA_DIR / "data_types.csv", index=False)
raw_df.head(25).to_csv(DATA_DIR / "example_raw_rows.csv", index=False)
df.head(25).to_csv(DATA_DIR / "example_engineered_rows.csv", index=False)


def records_json_safe(frame):
    temp = frame.copy()
    for c in temp.columns:
        if pd.api.types.is_datetime64_any_dtype(temp[c]):
            temp[c] = temp[c].dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
    temp = temp.replace([np.inf, -np.inf], np.nan)
    return json.loads(temp.to_json(orient="records", date_format="iso"))


# Detailed browser dataset plus a lighter version for smooth animation.
web_cols = [
    "point_id", "time", "elapsed_time_s", "elapsed_time_min",
    "latitude", "longitude",
    "elevation_m", "elevation_smooth_m",
    "segment_distance_m", "analysis_segment_distance_m",
    "cumulative_distance_km", "displacement_from_start_m",
    "bearing_deg", "grade_smooth_pct",
    "cumulative_elevation_gain_m",
    "speed_kmh_raw", "speed_smooth_kmh",
    "velocity_east_m_s", "velocity_north_m_s",
    "acceleration_smooth_m_s2", "vertical_speed_m_s",
    "is_moving", "speed_outlier_flag",
]
web_full = df[[c for c in web_cols if c in df.columns]].copy()

web_step = max(1, int(np.ceil(len(web_full) / 1800)))
web_track = web_full.iloc[::web_step].copy()
if web_track.index[-1] != web_full.index[-1]:
    web_track = pd.concat([web_track, web_full.tail(1)])

summary_payload = {
    "source_file": GPX_PATH.name,
    "activity_name": gpx_metadata.get("activity_name"),
    "has_time": HAS_REAL_TIME,
    "analysis_clock_available": HAS_TIME,
    "has_real_gpx_time": HAS_REAL_TIME,
    "time_is_synthetic": TIME_IS_SYNTHETIC,
    "time_source": TIME_SOURCE,
    "time_series_note": TIME_SERIES_NOTE,
    "master_slider": "elapsed_time" if HAS_REAL_TIME else "distance",
    "track_points": int(len(df)),
    "distance_km": distance_km,
    "raw_distance_km": raw_distance_km,
    "elevation_gain_m": float(df["elevation_gain_m"].sum()),
    "elevation_loss_m": float(df["elevation_loss_m"].sum()),
    "min_elevation_m": float(df["elevation_smooth_m"].min()),
    "max_elevation_m": float(df["elevation_smooth_m"].max()),
    "max_displacement_km": float(df["displacement_from_start_m"].max() / 1000.0),
    "duration_s": duration_s if HAS_REAL_TIME else None,
    "moving_time_s": moving_time_s if HAS_REAL_TIME else None,
    "average_speed_kmh": avg_speed if HAS_REAL_TIME else None,
    "average_moving_speed_kmh": avg_moving_speed if HAS_REAL_TIME else None,
    "max_speed_kmh": float(df["speed_smooth_kmh"].max()) if HAS_REAL_TIME else None,
    "speed_grade_correlation": speed_grade_corr if HAS_REAL_TIME and np.isfinite(speed_grade_corr) else None,
}

chart_catalog = [
    {"id":"01_missing_values","title":"Missing values","why":"Shows whether required source fields are absent.","x":"field","y":"missing count","available":True},
    {"id":"02_sampling_interval","title":"Sampling interval","why":"Checks how regularly GPS points were recorded.","x":"seconds","y":"frequency","available":HAS_REAL_TIME},
    {"id":"03_route_shape","title":"Route shape","why":"Validates spatial order and overall geometry before using the map.","x":"longitude","y":"latitude","available":True},
    {"id":"04_distance_vs_time","title":"Distance vs time","why":"Cumulative travelled distance must increase or stay flat.","x":"elapsed time","y":"distance","available":HAS_REAL_TIME},
    {"id":"05_displacement_vs_time","title":"Displacement vs time","why":"Unlike cumulative distance, straight-line displacement from the start can decrease.","x":"elapsed time","y":"displacement","available":HAS_REAL_TIME},
    {"id":"06_elevation_vs_distance","title":"Elevation profile","why":"Shows the terrain encountered along the route.","x":"distance","y":"elevation","available":True},
    {"id":"07_elevation_vs_time","title":"Elevation vs time","why":"Shows when climbing and descending happened.","x":"elapsed time","y":"elevation","available":HAS_REAL_TIME},
    {"id":"08_speed_vs_time","title":"Speed vs time","why":"Shows the magnitude of velocity through the ride.","x":"elapsed time","y":"speed","available":HAS_REAL_TIME},
    {"id":"09_velocity_components_vs_time","title":"Velocity components","why":"Uses bearing to separate motion into east and north components.","x":"elapsed time","y":"velocity component","available":HAS_REAL_TIME},
    {"id":"10_acceleration_vs_time","title":"Acceleration vs time","why":"Shows changes in the magnitude of velocity.","x":"elapsed time","y":"acceleration","available":HAS_REAL_TIME},
    {"id":"11_gradient_vs_distance","title":"Gradient vs distance","why":"Shows how steepness changes around the route.","x":"distance","y":"gradient","available":True},
    {"id":"12_speed_and_gradient_vs_time","title":"Speed + gradient vs time","why":"Places terrain and speed on the same time axis for comparison.","x":"elapsed time","y":"speed / gradient","available":HAS_REAL_TIME},
    {"id":"13_speed_vs_gradient_scatter","title":"Speed vs gradient","why":"Explores association between terrain and speed without claiming causation.","x":"gradient","y":"speed","available":HAS_REAL_TIME},
    {"id":"14_speed_distribution","title":"Speed distribution","why":"Summarises centre and spread of observed cycling speed.","x":"speed","y":"frequency","available":HAS_REAL_TIME},
    {"id":"15_cumulative_elevation_gain","title":"Cumulative climbing","why":"Shows how total ascent accumulates across the ride.","x":"distance","y":"cumulative gain","available":True},
    {"id":"16_elevation_and_gradient","title":"Elevation + gradient","why":"Separates absolute height from steepness.","x":"distance","y":"elevation / gradient","available":True},
    {"id":"17_segment_distance_distribution","title":"GPS segment distances","why":"Useful EDA for sampling behaviour and large GPS jumps.","x":"segment distance","y":"frequency","available":True},
    {"id":"18_acceleration_distribution","title":"Acceleration distribution","why":"Summarises the variability of estimated acceleration.","x":"acceleration","y":"frequency","available":HAS_REAL_TIME},
]

code_walkthrough = [
    {
        "title":"Parse GPX/XML",
        "purpose":"Turn XML track points into rows while handling XML namespaces.",
        "code":'''def local_name(tag):\n    return tag.split("}")[-1].lower()\n\nfor el in root.iter():\n    if local_name(el.tag) == "trkpt":\n        lat = float(el.attrib["lat"])\n        lon = float(el.attrib["lon"])''',
        "explanation":"A GPX is XML. Each trkpt becomes one observation in a pandas DataFrame."
    },
    {
        "title":"Haversine distance",
        "purpose":"Convert consecutive latitude/longitude observations into metres travelled.",
        "code":'''a = sin²(Δlat/2) + cos(lat1)·cos(lat2)·sin²(Δlon/2)\nc = 2·atan2(√a, √(1-a))\ndistance = EarthRadius · c''',
        "explanation":"Latitude/longitude are angles, so direct subtraction is not a distance in metres."
    },
    {
        "title":"Cumulative distance",
        "purpose":"Build the monotonically increasing distance series.",
        "code":'''df["cumulative_distance_km"] = (\n    df["analysis_segment_distance_m"].cumsum() / 1000\n)''',
        "explanation":"Every segment contributes a non-negative distance. The cumulative total can therefore increase or remain flat, never fall."
    },
    {
        "title":"Speed / velocity magnitude",
        "purpose":"Calculate speed when genuine timestamps exist.",
        "code":'''speed_m_s = segment_distance_m / delta_time_s\nspeed_kmh = speed_m_s * 3.6''',
        "explanation":"Speed is the magnitude of velocity. Direction is retained separately through bearing."
    },
    {
        "title":"Velocity components",
        "purpose":"Use speed + bearing to keep directional information.",
        "code":'''v_east = speed_m_s * sin(bearing)\nv_north = speed_m_s * cos(bearing)''',
        "explanation":"This is closer to the vector idea of velocity than a speed-only graph."
    },
    {
        "title":"Acceleration",
        "purpose":"Estimate change in velocity magnitude per second.",
        "code":'''acceleration = speed_m_s.diff() / delta_time_s''',
        "explanation":"Differentiation amplifies GPS noise, so the project retains raw values and uses robust smoothing for interpretation."
    },
    {
        "title":"Gradient",
        "purpose":"Measure steepness from elevation change relative to distance.",
        "code":'''gradient_pct = 100 * rise_m / run_m''',
        "explanation":"Elevation is how high the rider is; gradient is how quickly height changes with distance."
    },
    {
        "title":"SQLite storage",
        "purpose":"Persist raw, engineered and summary tables outside pandas memory.",
        "code":'''df.to_sql("engineered_track_points", conn, if_exists="replace", index=False)''',
        "explanation":"This creates a reproducible database layer that can be queried with standard SQL."
    },
]

website_story = {
    "headline":"One bicycle ride, from raw GPS points to an explorable data product.",
    "story_points": story_points,
    "concepts":[
        {"title":"EDA before calculation","text":"The project first checks structure, types, missingness, duplicates, coordinates and timestamps before deriving new measures."},
        {"title":"Raw vs engineered data","text":"Latitude, longitude, elevation and time come from the GPX; distance, gradient, speed, velocity components and acceleration are calculated features."},
        {"title":"Distance is not displacement","text":"Cumulative travelled distance never decreases. Displacement from the starting point can decrease if the cyclist rides back towards the start."},
        {"title":"Speed is not the whole velocity vector","text":"The speed-time graph shows velocity magnitude. Bearing allows east/north velocity components to be estimated."},
        {"title":"Smoothing is explicit","text":"GPS-derived speed, elevation and acceleration can be noisy. Raw values are retained while smoothed values are used for clearer interpretation."},
        {"title":"Flags are not automatic deletions","text":"Potential outliers are marked for investigation rather than silently removed."},
        {"title":"Cumulative gain is not altitude","text":"Current elevation can rise and fall. Cumulative elevation gain is the sum of positive climbing and therefore only increases."},
        {"title":"Missing time remains documented","text":"If timestamps are absent, the raw audit still reports them as missing. The zero-origin analysis clock is stored separately as a transparent computational fallback."},
        {"title":"One analytical pipeline","text":"Python performs the calculations once. CSV, SQLite, JSON, GeoJSON, charts and HTML all come from the same processed data."},
    ]
}

dashboard_payload = {
    "summary": summary_payload,
    "time_provenance": TIME_PROVENANCE,
    "metadata": gpx_metadata,
    "story": website_story,
    "chart_catalog": chart_catalog,
    "code_walkthrough": code_walkthrough,
    "track": records_json_safe(web_track),
    "track_full": records_json_safe(web_full),
    "kilometres": records_json_safe(km_summary),
    "minutes": records_json_safe(minute_summary) if not minute_summary.empty else [],
    "quality": records_json_safe(data_quality),
    "data_types": records_json_safe(dtype_table),
    "raw_dictionary": records_json_safe(raw_dictionary),
    "engineered_dictionary": records_json_safe(engineered_dictionary),
    "raw_stats": records_json_safe(raw_stats.reset_index().rename(columns={"index":"variable"})),
    "engineered_stats": records_json_safe(descriptive_stats.reset_index().rename(columns={"index":"variable"})),
    "raw_example": records_json_safe(raw_df.head(10)),
    "engineered_example": records_json_safe(df.head(10)),
    "sql_queries": [{"name":k, "sql":v.strip()} for k,v in queries.items()],
    "sql_results": {k: records_json_safe(v) for k,v in query_results.items()},
}

# Structured JSON exports
exports = {
    "project_summary.json": summary_payload,
    "dashboard_data.json": dashboard_payload,
    "chart_catalog.json": chart_catalog,
    "code_walkthrough.json": code_walkthrough,
    "website_story.json": website_story,
    "data_quality.json": records_json_safe(data_quality),
    "data_types.json": records_json_safe(dtype_table),
    "kilometre_summary.json": records_json_safe(km_summary),
    "minute_summary.json": records_json_safe(minute_summary) if not minute_summary.empty else [],
    "track_points_web.json": records_json_safe(web_track),
}
for filename, obj in exports.items():
    (JSON_DIR / filename).write_text(json.dumps(obj, indent=2), encoding="utf-8")

# Full browser data is kept separately because it can be larger.
(JSON_DIR / "track_points_full.json").write_text(
    json.dumps(records_json_safe(web_full), indent=2),
    encoding="utf-8"
)

# GeoJSON route
geojson = {
    "type":"FeatureCollection",
    "features":[{
        "type":"Feature",
        "properties":summary_payload,
        "geometry":{
            "type":"LineString",
            "coordinates":[
                [float(lon), float(lat)]
                for lat, lon in zip(web_track["latitude"], web_track["longitude"])
            ]
        }
    }]
}
(JSON_DIR / "route.geojson").write_text(json.dumps(geojson, indent=2), encoding="utf-8")

# JavaScript data file for easy GitHub Pages integration.
(WEBSITE_DIR / "project_data.js").write_text(
    "window.STRAVA_RIDE_DATA = " + json.dumps(dashboard_payload, separators=(",", ":")) + ";",
    encoding="utf-8"
)
(WEBSITE_DIR / "project_data.json").write_text(
    json.dumps(dashboard_payload, indent=2),
    encoding="utf-8"
)

print("Complete structured web dataset exported.")


## Part I — Build the Explanatory Interactive Ride Explorer

The exported website is intentionally much more than a dashboard.

It contains:

- a **full-width map** with a moving bicycle;
- a master slider using **elapsed time when available**, otherwise distance;
- synchronized ride metrics;
- interactive distance, displacement, speed, acceleration, elevation and gradient charts;
- EDA and data-quality tables;
- raw vs engineered data examples;
- a chart catalogue explaining what every diagram means;
- a critical-code walkthrough;
- SQL examples and results;
- a narrative "What happened on this ride?" section;
- a transparent explanation of any unavailable measurements.


In [ ]:
follow_execution()

payload = json.dumps(dashboard_payload, separators=(",", ":"))

html = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Strava GPX — Data Analysis Explorer</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>
:root{--text:#0f172a;--muted:#64748b;--border:#dbe2ea;--soft:#f8fafc;--dark:#172033;--accent:#2563eb}
*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;font-family:Inter,system-ui,-apple-system,"Segoe UI",sans-serif;color:var(--text);background:#fff;line-height:1.45}.wrap{width:min(1500px,100%);margin:auto;padding:18px}.hero h1{margin:0 0 7px;font-size:clamp(1.8rem,4vw,3rem)}.hero p{max-width:1000px;color:#334155;margin:.3rem 0}.eyebrow{text-transform:uppercase;letter-spacing:.08em;font-size:.74rem;font-weight:800;color:var(--muted)}nav{display:flex;gap:7px;flex-wrap:wrap;margin:16px 0}nav a{border:1px solid var(--border);border-radius:999px;padding:7px 11px;text-decoration:none;color:inherit;font-size:.82rem}.banner{padding:12px 14px;border:1px solid var(--border);border-radius:12px;background:var(--soft);margin:12px 0}.cards,.live{display:grid;grid-template-columns:repeat(6,minmax(0,1fr));gap:8px;margin:12px 0}.card{border:1px solid var(--border);border-radius:11px;padding:10px;background:#fff;min-width:0}.card span{display:block;color:var(--muted);font-size:.67rem;text-transform:uppercase;letter-spacing:.04em;font-weight:800}.card strong{display:block;margin-top:3px;font-size:1.05rem;overflow:hidden;text-overflow:ellipsis}.section{margin:24px 0}.section>h2{margin:0 0 5px}.section-intro{margin:0 0 12px;color:#475569;max-width:1050px}.mapbox,.chartbox,.panel{border:1px solid var(--border);border-radius:14px;overflow:hidden;background:#fff;margin-bottom:12px}.head{display:flex;justify-content:space-between;gap:10px;padding:10px 12px;border-bottom:1px solid var(--border)}.head span{font-size:.77rem;color:var(--muted)}#map{height:650px;width:100%}.controls{display:grid;grid-template-columns:auto minmax(0,1fr) auto;gap:10px;align-items:center;padding:11px;border:1px solid var(--border);border-radius:12px;background:var(--soft);margin:12px 0}.controls input{width:100%}.controls button{min-width:48px;min-height:42px}.chart{height:320px;padding:8px 10px 12px}.chart canvas{width:100%!important;height:100%!important}.grid2{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px}.grid3{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px}.story{display:grid;gap:8px}.story-item{border-left:4px solid var(--border);padding:8px 12px;background:var(--soft)}.pipeline{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:8px}.step{border:1px solid var(--border);border-radius:10px;padding:11px;background:#fff}.step strong{display:block;margin-bottom:4px}.tablewrap{overflow:auto;border:1px solid var(--border);border-radius:10px}.tablewrap table{width:100%;border-collapse:collapse;font-size:.78rem}.tablewrap th,.tablewrap td{padding:8px;border-bottom:1px solid var(--border);text-align:right;white-space:nowrap}.tablewrap th:first-child,.tablewrap td:first-child{text-align:left}.tablewrap th{background:var(--soft);position:sticky;top:0}.catalog{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px}.catalog article{border:1px solid var(--border);border-radius:10px;padding:11px}.catalog h3{font-size:.95rem;margin:0 0 4px}.catalog p{font-size:.82rem;color:#475569;margin:4px 0}.tag{display:inline-block;border:1px solid var(--border);border-radius:999px;padding:3px 7px;font-size:.68rem;color:var(--muted)}details{border:1px solid var(--border);border-radius:10px;padding:10px 12px;margin:8px 0}summary{cursor:pointer;font-weight:750}pre{overflow:auto;background:#0f172a;color:#e2e8f0;padding:12px;border-radius:8px;font-size:.78rem;white-space:pre-wrap}.bike{font-size:28px}.unavailable{padding:18px;text-align:center;color:#64748b;background:var(--soft);border-radius:10px}.small{font-size:.8rem;color:var(--muted)}@media(max-width:1000px){.cards,.live{grid-template-columns:repeat(3,minmax(0,1fr))}.grid2,.grid3,.catalog{grid-template-columns:1fr}.pipeline{grid-template-columns:1fr}#map{height:520px}}@media(max-width:600px){.wrap{padding:12px}.cards,.live{grid-template-columns:repeat(2,minmax(0,1fr))}#map{height:400px}.controls{grid-template-columns:auto 1fr}.controls strong{grid-column:1/-1;text-align:center}}
</style>
</head>
<body><div class="wrap">
<section class="hero" id="top"><div class="eyebrow">GPX · EDA · KINEMATICS · SQL · LEAFLET · WEB DATA</div><h1>Afternoon Ride — Interactive Data Analysis</h1><p id="heroStory"></p></section>
<nav><a href="#ride">Ride Explorer</a><a href="#physics">Motion</a><a href="#terrain">Terrain</a><a href="#eda">EDA</a><a href="#engineering">Raw → Engineered</a><a href="#code">Code</a><a href="#sql">SQL</a><a href="#exports">Outputs</a></nav>
<div class="banner" id="timeBanner"></div>
<div class="cards"><div class="card"><span>Distance</span><strong id="sDistance"></strong></div><div class="card"><span>Duration</span><strong id="sDuration"></strong></div><div class="card"><span>Avg moving speed</span><strong id="sAverage"></strong></div><div class="card"><span>Max speed</span><strong id="sMax"></strong></div><div class="card"><span>Elevation gain</span><strong id="sGain"></strong></div><div class="card"><span>Track points</span><strong id="sPoints"></strong></div></div>

<section class="section" id="ride"><h2>Ride Explorer</h2><p class="section-intro">One master slider controls the bicycle, travelled route and live measurements. With timestamps it represents elapsed time; without them it represents progress through cumulative distance.</p><div class="mapbox"><div class="head"><strong>Full-width interactive route</strong><span>Zoom · pan · moving bicycle</span></div><div id="map"></div></div><div class="controls"><button id="play" type="button">▶</button><input id="slider" type="range" min="0" max="1000" value="0" step="1"><strong id="masterReadout">0</strong></div><div class="live"><div class="card"><span>Distance</span><strong id="lDistance">—</strong></div><div class="card"><span>Speed</span><strong id="lSpeed">—</strong></div><div class="card"><span>Acceleration</span><strong id="lAccel">—</strong></div><div class="card"><span>Elevation</span><strong id="lElevation">—</strong></div><div class="card"><span>Gradient</span><strong id="lGrade">—</strong></div><div class="card"><span>Climbing</span><strong id="lGain">—</strong></div></div><div class="story" id="storyPoints"></div></section>

<section class="section" id="physics"><h2>Motion: distance, displacement, velocity and acceleration</h2><p class="section-intro">Distance travelled and displacement are deliberately separated. Cumulative distance never falls; displacement from the start may fall when the cyclist comes back towards the start. Speed is velocity magnitude; bearing is used to retain directional information.</p><div id="timeCharts"></div></section>

<section class="section" id="terrain"><h2>Terrain and effort</h2><p class="section-intro">Elevation describes height. Gradient describes how quickly that height changes with distance. The project compares terrain with speed only as an exploratory association, not as proof of causation.</p><div class="grid2"><div class="chartbox"><div class="head"><strong>Elevation vs distance</strong><span>Recorded terrain profile</span></div><div class="chart"><canvas id="elevationChart"></canvas></div></div><div class="chartbox"><div class="head"><strong>Gradient vs distance</strong><span>Steepness</span></div><div class="chart"><canvas id="gradientChart"></canvas></div></div></div><div id="speedGradientArea"></div></section>

<section class="section" id="eda"><h2>EDA — Can we trust the source?</h2><p class="section-intro">Before modelling, the notebook checks what exists, what is missing, the data types, coordinate validity, duplicates, timestamp availability, sampling behaviour and potential outliers.</p><div class="cards" id="qualityCards"></div><div class="grid2"><div><h3>Data types</h3><div class="tablewrap"><table id="typesTable"></table></div></div><div><h3>Data-quality checks</h3><div class="tablewrap"><table id="qualityTable"></table></div></div></div><h3>Descriptive statistics</h3><div class="tablewrap"><table id="statsTable"></table></div></section>

<section class="section" id="engineering"><h2>Raw → Engineered</h2><p class="section-intro">The most important technical idea is knowing what the source supplied and what the analysis created.</p><div class="pipeline"><div class="step"><strong>1. Raw GPX</strong>Latitude · longitude · elevation · timestamp where present</div><div class="step"><strong>2. Validate</strong>Types · missingness · duplicates · coordinate checks · time checks</div><div class="step"><strong>3. Engineer</strong>Distance · displacement · bearing · gradient · speed · velocity · acceleration</div><div class="step"><strong>4. Persist</strong>CSV · SQLite · SQL · JSON · GeoJSON</div><div class="step"><strong>5. Communicate</strong>Charts · Leaflet · synchronized browser app</div></div><div class="grid2"><div><h3>Raw example</h3><div class="tablewrap"><table id="rawTable"></table></div></div><div><h3>Engineered example</h3><div class="tablewrap"><table id="engTable"></table></div></div></div><h3>Chart catalogue</h3><div class="catalog" id="catalog"></div></section>

<section class="section" id="code"><h2>Critical code walkthrough</h2><p class="section-intro">These are the small pieces of code Erik should be able to explain. The point is understanding the transformation, not memorising every line.</p><div id="codeList"></div></section>

<section class="section" id="sql"><h2>SQLite / SQL</h2><p class="section-intro">The processed data is written to a local relational database so the project can be queried after the pandas analysis is complete.</p><div id="sqlList"></div></section>

<section class="section" id="exports"><h2>What the notebook exports</h2><p class="section-intro">The website is a presentation layer over reusable outputs, not a second calculation engine.</p><div class="grid3"><div class="step"><strong>Data</strong>Raw + engineered CSV, data dictionary, quality checks, kilometre/minute summaries.</div><div class="step"><strong>Analysis</strong>SQLite database, SQL query pack, descriptive statistics, PNG + SVG figures.</div><div class="step"><strong>Web</strong>JSON, GeoJSON, JavaScript payload, standalone Leaflet map and this explanatory app.</div></div><p class="small">Source provenance is retained by copying the original GPX into the output bundle.</p></section>
</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script><script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.7/dist/chart.umd.min.js"></script>
<script>
const D=__PAYLOAD__,S=D.summary,T=D.track,H=!!S.has_time;
const [heroStory,timeBanner,sDistance,sDuration,sAverage,sMax,sGain,sPoints,storyPoints,masterReadout,lDistance,lSpeed,lAccel,lElevation,lGrade,lGain,play,qualityCards,catalog,codeList,sqlList]=['heroStory','timeBanner','sDistance','sDuration','sAverage','sMax','sGain','sPoints','storyPoints','masterReadout','lDistance','lSpeed','lAccel','lElevation','lGrade','lGain','play','qualityCards','catalog','codeList','sqlList'].map(id=>document.getElementById(id));
const fmt=(x,d=1)=>Number.isFinite(Number(x))?Number(x).toLocaleString(undefined,{minimumFractionDigits:d,maximumFractionDigits:d}):"—";
const fmtTime=s=>{if(!Number.isFinite(Number(s)))return "—";s=Math.max(0,+s);const h=Math.floor(s/3600),m=Math.floor((s%3600)/60),q=Math.floor(s%60);return h?`${h}:${String(m).padStart(2,"0")}:${String(q).padStart(2,"0")}`:`${m}:${String(q).padStart(2,"0")}`};
const table=(id,rows,max=40)=>{const el=document.getElementById(id);if(!rows?.length){el.innerHTML='<tbody><tr><td>No data</td></tr></tbody>';return}const cols=Object.keys(rows[0]);el.innerHTML='<thead><tr>'+cols.map(c=>`<th>${c}</th>`).join('')+'</tr></thead><tbody>'+rows.slice(0,max).map(r=>'<tr>'+cols.map(c=>`<td>${r[c]??''}</td>`).join('')+'</tr>').join('')+'</tbody>'};
heroStory.textContent=D.story.headline;timeBanner.innerHTML=REAL?'<strong>Time series available.</strong> These measures use genuine GPX timestamps.':(SYN?'<strong>Estimated time series.</strong> The GPX timestamps were stripped. The known Strava start time and moving duration are used to build a uniform analysis clock across point order. Speed and acceleration are therefore estimates, clearly labelled as reconstructed.':'<strong>Time series unavailable.</strong>');
sDistance.textContent=fmt(S.distance_km,2)+' km';sDuration.textContent=H?fmtTime(S.duration_s):'No timestamps';sAverage.textContent=H?fmt(S.average_moving_speed_kmh,1)+' km/h':'—';sMax.textContent=H?fmt(S.max_speed_kmh,1)+' km/h':'—';sGain.textContent=fmt(S.elevation_gain_m,0)+' m';sPoints.textContent=(S.track_points||0).toLocaleString();
storyPoints.innerHTML=D.story.story_points.map(x=>`<div class="story-item">${x}</div>`).join('');
const route=T.map(p=>[+p.latitude,+p.longitude]);const map=L.map('map',{scrollWheelZoom:true});L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',{maxZoom:19,attribution:'Tiles &copy; Esri'}).addTo(map);const base=L.polyline(route,{weight:6,opacity:.55}).addTo(map);if(route.length)map.fitBounds(base.getBounds(),{padding:[24,24]});const travelled=L.polyline(route.length?[route[0]]:[],{weight:7,opacity:.95}).addTo(map);const bike=L.marker(route[0],{icon:L.divIcon({className:'',html:'<div class="bike">🚲</div>',iconSize:[36,36],iconAnchor:[18,18]})}).addTo(map);
const master=T.map(p=>H?+p.elapsed_time_s:+p.cumulative_distance_km),masterMax=master[master.length-1]||1;function nearest(v){let l=0,h=T.length-1;while(l<h){const m=Math.floor((l+h)/2);if(master[m]<v)l=m+1;else h=m}return l}const slider=document.getElementById('slider');function update(progress){const target=masterMax*progress,i=nearest(target),p=T[i];travelled.setLatLngs(route.slice(0,i+1));bike.setLatLng(route[i]);masterReadout.textContent=H?fmtTime(p.elapsed_time_s):fmt(p.cumulative_distance_km,2)+' km';lDistance.textContent=fmt(p.cumulative_distance_km,2)+' km';lSpeed.textContent=H?fmt(p.speed_smooth_kmh,1)+' km/h':'—';lAccel.textContent=H?fmt(p.acceleration_smooth_m_s2,2)+' m/s²':'—';lElevation.textContent=fmt(p.elevation_smooth_m,1)+' m';lGrade.textContent=fmt(p.grade_smooth_pct,1)+'%';lGain.textContent=fmt(p.cumulative_elevation_gain_m,0)+' m';for(const item of cursorCharts){item.chart.options.plugins.rideCursor.xValue=item.axis==='time'?+p.elapsed_time_min:+p.cumulative_distance_km;item.chart.draw()}}slider.addEventListener('input',()=>update(+slider.value/1000));let timer=null;play.addEventListener('click',()=>{if(timer){clearInterval(timer);timer=null;play.textContent='▶'}else{play.textContent='❚❚';timer=setInterval(()=>{let v=+slider.value+3;if(v>1000)v=0;slider.value=v;update(v/1000)},45)}});

// ------------------------------------------------------------
// INTERACTIVE CHART CONTROLS
// ------------------------------------------------------------
// Hover: nearest point is easy to inspect even though normal point dots are hidden.
// Click/tap: select that graph position and move the master ride slider + map there.
// Play: the existing master slider continues to move the map and synchronized cursors.

const nearestTrackIndex=(axis,xValue)=>{
  let best=0, bestGap=Infinity;
  for(let i=0;i<T.length;i++){
    const v=axis==='time'?+T[i].elapsed_time_min:+T[i].cumulative_distance_km;
    if(!Number.isFinite(v))continue;
    const gap=Math.abs(v-xValue);
    if(gap<bestGap){best=i;bestGap=gap}
  }
  return best;
};

const selectTrackIndex=(idx)=>{
  idx=Math.max(0,Math.min(T.length-1,idx));
  const fraction=T.length>1?idx/(T.length-1):0;
  slider.value=Math.round(fraction*1000);
  update(fraction);
};

const interactiveLineOptions=(axis,xTitle,yTitle)=>({
  responsive:true,
  maintainAspectRatio:false,
  animation:false,
  interaction:{mode:'nearest',intersect:false,axis:'x'},
  elements:{point:{radius:0,hitRadius:14,hoverRadius:5}},
  plugins:{
    legend:{display:false},
    tooltip:{
      enabled:true,
      mode:'nearest',
      intersect:false,
      callbacks:{
        title(items){
          if(!items.length)return '';
          const x=items[0].parsed.x;
          return axis==='time'
            ? `Time: ${Number(x).toFixed(2)} min`
            : `Distance: ${Number(x).toFixed(2)} km`;
        },
        label(ctx){
          const y=ctx.parsed.y;
          return `${ctx.dataset.label||yTitle}: ${Number(y).toFixed(2)}`;
        },
        afterBody(items){
          if(!items.length)return [];
          const x=items[0].parsed.x;
          const idx=nearestTrackIndex(axis,x);
          const p=T[idx];
          const lines=[
            `Elevation: ${fmt(p.elevation_smooth_m,1)} m`,
            `Gradient: ${fmt(p.grade_smooth_pct,1)}%`
          ];
          if(H){
            lines.push(`Speed: ${fmt(p.speed_smooth_kmh,1)} km/h`);
            lines.push(`Elapsed: ${fmt(p.elapsed_time_min,1)} min`);
          }
          return lines;
        }
      }
    },
    rideCursor:{xValue:0}
  },
  onClick(evt,elements,chart){
    const pts=chart.getElementsAtEventForMode(evt,'nearest',{intersect:false},true);
    if(!pts.length)return;
    const x=chart.data.datasets[pts[0].datasetIndex].data[pts[0].index].x;
    selectTrackIndex(nearestTrackIndex(axis,+x));
  },
  scales:{
    x:{type:'linear',title:{display:true,text:xTitle}},
    y:{title:{display:true,text:yTitle}}
  }
});

const cursorPlugin={id:'rideCursor',afterDraw(c,a,o){const v=+(o?.xValue);if(!Number.isFinite(v))return;const x=c.scales.x.getPixelForValue(v);if(!Number.isFinite(x))return;const q=c.ctx,A=c.chartArea;q.save();q.strokeStyle='#2563eb';q.setLineDash([5,5]);q.beginPath();q.moveTo(x,A.top);q.lineTo(x,A.bottom);q.stroke();q.restore()}};Chart.register(cursorPlugin);const cursorCharts=[];
const registerCursor=(chart,axis)=>cursorCharts.push({chart,axis});
const mkLine=(canvas,data,xTitle,yTitle,axis='distance',label=yTitle)=>{const c=new Chart(canvas,{type:'line',data:{datasets:[{label,data,parsing:false,pointRadius:0,borderWidth:2,tension:.1}]},options:interactiveLineOptions(axis,xTitle,yTitle)});registerCursor(c,axis);return c};
const timeArea=document.getElementById('timeCharts');if(H){timeArea.innerHTML='<div class="chartbox"><div class="head"><strong>Distance vs time</strong><span>Monotonic travelled distance</span></div><div class="chart"><canvas id="distanceChart"></canvas></div></div><div class="chartbox"><div class="head"><strong>Displacement from start vs time</strong><span>May rise or fall</span></div><div class="chart"><canvas id="dispChart"></canvas></div></div><div class="chartbox"><div class="head"><strong>Speed vs time</strong><span>Velocity magnitude</span></div><div class="chart"><canvas id="speedChart"></canvas></div></div><div class="chartbox"><div class="head"><strong>Acceleration vs time</strong><span>Approximate tangential acceleration</span></div><div class="chart"><canvas id="accChart"></canvas></div></div><div class="chartbox"><div class="head"><strong>East/North velocity components</strong><span>Direction retained through bearing</span></div><div class="chart"><canvas id="velChart"></canvas></div></div>';const tx='Elapsed time (minutes)';mkLine(document.getElementById('distanceChart'),T.map(p=>({x:+p.elapsed_time_min,y:+p.cumulative_distance_km})),tx,'Distance (km)','time');mkLine(document.getElementById('dispChart'),T.map(p=>({x:+p.elapsed_time_min,y:+p.displacement_from_start_m/1000})),tx,'Displacement (km)','time');mkLine(document.getElementById('speedChart'),T.map(p=>({x:+p.elapsed_time_min,y:+p.speed_smooth_kmh})),tx,'Speed (km/h)','time');mkLine(document.getElementById('accChart'),T.map(p=>({x:+p.elapsed_time_min,y:+p.acceleration_smooth_m_s2})).filter(p=>Number.isFinite(p.y)),tx,'Acceleration (m/s²)','time');const vc=new Chart(document.getElementById('velChart'),{type:'line',data:{datasets:[{label:'East',data:T.map(p=>({x:+p.elapsed_time_min,y:+p.velocity_east_m_s})),parsing:false,pointRadius:0,borderWidth:1.5},{label:'North',data:T.map(p=>({x:+p.elapsed_time_min,y:+p.velocity_north_m_s})),parsing:false,pointRadius:0,borderWidth:1.5}]},options:{
responsive:true,maintainAspectRatio:false,animation:false,
interaction:{mode:'nearest',intersect:false,axis:'x'},
elements:{point:{radius:0,hitRadius:14,hoverRadius:5}},
plugins:{
  rideCursor:{xValue:0},
  tooltip:{
    enabled:true,mode:'nearest',intersect:false,
    callbacks:{
      title(items){return items.length?`Time: ${Number(items[0].parsed.x).toFixed(2)} min`:''},
      label(ctx){return `${ctx.dataset.label}: ${Number(ctx.parsed.y).toFixed(2)} m/s`}
    }
  }
},
onClick(evt,elements,chart){
  const pts=chart.getElementsAtEventForMode(evt,'nearest',{intersect:false},true);
  if(!pts.length)return;
  const x=chart.data.datasets[pts[0].datasetIndex].data[pts[0].index].x;
  selectTrackIndex(nearestTrackIndex('time',+x));
},
scales:{x:{type:'linear',title:{display:true,text:tx}},y:{title:{display:true,text:'Velocity component (m/s)'}}}
}});registerCursor(vc,'time')}else{timeArea.innerHTML='<div class="unavailable"><strong>Time-based motion charts are unavailable.</strong><br>This GPX has no complete timestamp series, so the project refuses to manufacture distance-time, speed-time or acceleration-time results.</div>'}
mkLine(document.getElementById('elevationChart'),T.map(p=>({x:+p.cumulative_distance_km,y:+p.elevation_smooth_m})),'Distance (km)','Elevation (m)','distance');mkLine(document.getElementById('gradientChart'),T.map(p=>({x:+p.cumulative_distance_km,y:+p.grade_smooth_pct})).filter(p=>Number.isFinite(p.y)),'Distance (km)','Gradient (%)','distance');
const sg=document.getElementById('speedGradientArea');if(H){sg.innerHTML='<div class="grid2"><div class="chartbox"><div class="head"><strong>Speed + gradient vs time</strong><span>Same horizontal axis</span></div><div class="chart"><canvas id="sgTime"></canvas></div></div><div class="chartbox"><div class="head"><strong>Speed vs gradient</strong><span>Exploratory association</span></div><div class="chart"><canvas id="sgScatter"></canvas></div></div></div>';const c=new Chart(document.getElementById('sgTime'),{type:'line',data:{datasets:[{label:'Speed',data:T.map(p=>({x:+p.elapsed_time_min,y:+p.speed_smooth_kmh})),parsing:false,pointRadius:0,borderWidth:2,yAxisID:'a'},{label:'Gradient',data:T.map(p=>({x:+p.elapsed_time_min,y:+p.grade_smooth_pct})).filter(p=>Number.isFinite(p.y)),parsing:false,pointRadius:0,borderWidth:1.3,borderDash:[6,4],yAxisID:'b'}]},options:{
responsive:true,maintainAspectRatio:false,animation:false,
interaction:{mode:'nearest',intersect:false,axis:'x'},
elements:{point:{radius:0,hitRadius:14,hoverRadius:5}},
plugins:{
  rideCursor:{xValue:0},
  tooltip:{
    enabled:true,mode:'nearest',intersect:false,
    callbacks:{
      title(items){return items.length?`Time: ${Number(items[0].parsed.x).toFixed(2)} min`:''},
      label(ctx){
        const unit=ctx.dataset.label==='Speed'?' km/h':'%';
        return `${ctx.dataset.label}: ${Number(ctx.parsed.y).toFixed(2)}${unit}`;
      }
    }
  }
},
onClick(evt,elements,chart){
  const pts=chart.getElementsAtEventForMode(evt,'nearest',{intersect:false},true);
  if(!pts.length)return;
  const x=chart.data.datasets[pts[0].datasetIndex].data[pts[0].index].x;
  selectTrackIndex(nearestTrackIndex('time',+x));
},
scales:{x:{type:'linear',title:{display:true,text:'Elapsed time (minutes)'}},a:{position:'left',title:{display:true,text:'Speed (km/h)'}},b:{position:'right',title:{display:true,text:'Gradient (%)'},grid:{drawOnChartArea:false}}}
}});registerCursor(c,'time');new Chart(document.getElementById('sgScatter'),{type:'scatter',data:{datasets:[{data:T.map(p=>({x:+p.grade_smooth_pct,y:+p.speed_smooth_kmh})).filter(p=>Number.isFinite(p.x)&&Number.isFinite(p.y)),pointRadius:2}]},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{x:{title:{display:true,text:'Gradient (%)'}},y:{title:{display:true,text:'Speed (km/h)'}}}}})}else{sg.innerHTML='<div class="unavailable">Speed-gradient comparison needs time-derived speed. Gradient remains available.</div>'}
const q=D.quality||[];qualityCards.innerHTML=q.slice(1,7).map(r=>`<div class="card"><span>${r.check}</span><strong>${r.count}</strong></div>`).join('');table('typesTable',D.data_types);table('qualityTable',D.quality);table('statsTable',D.engineered_stats,100);table('rawTable',D.raw_example,10);table('engTable',D.engineered_example,10);
catalog.innerHTML=D.chart_catalog.map(x=>`<article><h3>${x.title}</h3><span class="tag">${x.available?'available':'unavailable from source'}</span><p>${x.why}</p><p class="small">${x.x} → ${x.y}</p></article>`).join('');codeList.innerHTML=D.code_walkthrough.map(x=>`<details><summary>${x.title}</summary><p><strong>Purpose:</strong> ${x.purpose}</p><pre><code>${x.code.replace(/&/g,'&amp;').replace(/</g,'&lt;')}</code></pre><p>${x.explanation}</p></details>`).join('');sqlList.innerHTML=D.sql_queries.map(x=>`<details><summary>${x.name}</summary><pre><code>${x.sql.replace(/&/g,'&amp;').replace(/</g,'&lt;')}</code></pre></details>`).join('');update(0);
</script></body></html>'''.replace("__PAYLOAD__", payload)

DASHBOARD_PATH = WEBSITE_DIR / "ride_explorer.html"
DASHBOARD_PATH.write_text(html, encoding="utf-8")
shutil.copy2(MAP_PATH, WEBSITE_DIR / "ride_full_width_map.html")
print("Explanatory website:", DASHBOARD_PATH)



## Interactive Ride Explorer — inside Colab and as a standalone website

The project now deliberately provides **both versions** from the same generated dashboard:

1. **Interactive inside this Colab notebook** — Play/Pause, master slider, moving ride position, hover tooltips, graph selection and synchronized chart cursors.
2. **Standalone `ride_explorer.html`** — the same dashboard saved in the website bundle for the portfolio site.

This means the analytical notebook demonstrates the interaction directly, while the employer-facing website receives the polished standalone version.



### Standalone Ride Explorer

The full `ride_explorer.html` is still generated for the website bundle.

Inside Colab, the notebook now uses a **native Plotly interactive dashboard** instead of embedding the website iframe. This avoids the blank/partial Ride Explorer behaviour that can occur inside Colab while keeping the website version intact.


## Part J — Reports, Website Assets, Manifest + Final ZIP

The final ZIP is deliberately organised so it can be handed directly back to ChatGPT for the live Projects website.


In [ ]:
follow_execution()

# Copy static figures into the website bundle as well, while retaining the master figure folder.
WEB_FIGURES_DIR = WEBSITE_DIR / "figures"
WEB_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
for p in FIGURES_DIR.iterdir():
    if p.is_file():
        shutil.copy2(p, WEB_FIGURES_DIR / p.name)

technical_report = f"""# {ACTIVITY_TITLE} — Technical Report

## Project pipeline

Raw GPX → XML parsing → pandas → EDA → feature engineering → interactive Plotly analysis + static plotting → SQLite / SQL → PNG/SVG → Leaflet → JSON/GeoJSON → explanatory browser app

## Source status

- Source file: `{GPX_PATH.name}`
- Track points: {len(df):,}
- Genuine GPX timestamp series: {HAS_REAL_TIME}
- Analysis clock available: {HAS_TIME}
- Placeholder time axis: {TIME_IS_SYNTHETIC}
- Time source: {TIME_SOURCE}
- Distance: {distance_km:.2f} km
- Pointwise smoothed cumulative elevation gain: {df['elevation_gain_m'].sum():.1f} m
- 50 m resampled-profile elevation gain: {elevation_gain_profile_m:.1f} m
- Highest smoothed elevation: {highest['elevation_smooth_m']:.1f} m

## Time handling

{'The GPX contains genuine timestamps, so time-dependent measures use observed point times.' if HAS_REAL_TIME else 'The GPX has no usable point timestamps. The raw missingness is preserved. A separate placeholder clock starts at 2000-01-01 00:00:00 UTC and advances by one second per point. Time-derived quantities are computational examples under that assumption, not measured ride performance.'}

## pandas learning

This project uses pandas to move from raw observations to an analytical dataset:

1. inspect with `head`, `tail`, `sample`, `dtypes` and `describe`;
2. test missingness and validity with boolean masks;
3. compare neighbouring rows with `shift` and `diff`;
4. create derived columns with vectorised arithmetic;
5. accumulate distance and climbing with `cumsum`;
6. smooth noisy signals with `rolling`;
7. summarise sections of the ride with `groupby` and `agg`.

## Plotting learning

The project deliberately keeps two visual layers: interactive Plotly dashboards for exploration and employer demonstration, and static Matplotlib PNG/SVG figures for reports and archival output. The interactive suite covers motion through time, terrain/climbing, distributions, relationships and data-quality diagnostics.

## Core analytical ideas

1. **EDA before modelling:** establish what the file actually contains.
2. **Raw vs engineered:** calculated features are kept distinct from source fields.
3. **Distance vs displacement:** travelled distance can only rise; displacement can fall.
4. **Smoothing:** raw values are preserved while smoothed values support interpretation.
5. **Missing time provenance:** a fallback analysis clock never changes the fact that original timestamps were absent.
6. **Persistence:** raw, engineered and summary tables are written to SQLite and queried with SQL.
7. **Presentation layer:** browser outputs use exported processed data rather than silently repeating the Python analysis.

## Website architecture

The bundle contains the Ride Explorer, route map, terrain analysis, EDA, raw-vs-engineered examples, plotting catalogue, critical-code walkthrough and SQL examples.
"""

REPORT_PATH = REPORT_DIR / "technical_report.md"
REPORT_PATH.write_text(technical_report, encoding="utf-8")

code_report = "# Critical Code Walkthrough\n\n" + "\n\n".join(
    f"## {x['title']}\n\n**Purpose:** {x['purpose']}\n\n```text\n{x['code']}\n```\n\n{x['explanation']}"
    for x in code_walkthrough
)
(REPORT_DIR / "critical_code_walkthrough.md").write_text(code_report, encoding="utf-8")

website_content = "# Website Story\n\n" + website_story["headline"] + "\n\n## Story of the Ride\n\n" + "\n".join(f"- {x}" for x in story_points) + "\n\n## Concepts\n\n" + "\n\n".join(f"### {x['title']}\n\n{x['text']}" for x in website_story["concepts"])
(REPORT_DIR / "website_content.md").write_text(website_content, encoding="utf-8")

interview_report = f"""# Interview Handover — Strava GPX Analysis

## Key line

I took a raw GPX file through validation, feature engineering, database storage, SQL, visualisation and an interactive browser presentation.

## Evidence

- {len(df):,} track points
- {distance_km:.2f} km analysed route
- EDA before feature engineering
- Haversine distance and bearing from latitude/longitude
- elevation smoothing, climbing and gradient
- {'genuine timestamp-derived speed, velocity components and acceleration' if HAS_REAL_TIME else 'a clearly labelled zero-origin placeholder clock used only to keep the pandas/plotting pipeline operational'}
- raw and engineered tables stored separately
- SQLite + SQL query layer
- reusable JSON/GeoJSON/CSV/PNG/SVG/HTML outputs
- Leaflet/Esri interactive map
- synchronized web Ride Explorer
- full-width interactive Plotly suite: one chart per row for motion, terrain, relationships, distributions and data-quality diagnostics

## Strong interview lesson

The most important part is not the cycling subject. It is the workflow: validate source data, preserve provenance, engineer only defensible features, identify exceptions, persist the result, query it and communicate it to another person.
"""
(REPORT_DIR / "interview_handover.md").write_text(interview_report, encoding="utf-8")

README = f"""# {ACTIVITY_TITLE} — Complete Website Bundle

## Open first

- `website/ride_explorer.html` — standalone interactive dashboard; the same dashboard is also embedded and demonstrated inside the Colab notebook
- `website/ride_full_width_map.html` — standalone analytical zoomable map
- `report/technical_report.md`
- `report/critical_code_walkthrough.md`
- `report/interview_handover.md`

## Website data

- `website/project_data.json`
- `website/project_data.js`
- `json/time_provenance.json`
- `json/dashboard_data.json`
- `json/chart_catalog.json`
- `json/code_walkthrough.json`
- `json/website_story.json`
- `json/route.geojson`

## Static graphics

Every analytical figure is exported to PNG + SVG under `figures/` and copied to `website/figures/`.

## Database

- `sql/ride_analysis.db`
- `sql/example_queries.sql`

## Source / provenance

The original GPX is retained under `source/`.

## Time provenance

Genuine timestamps present: {HAS_REAL_TIME}
Placeholder clock used: {TIME_IS_SYNTHETIC}
"""
(OUTPUT_DIR / "README_OUTPUTS.md").write_text(README, encoding="utf-8")

# Manifest written before ZIP so the website builder can see every asset.
manifest = []
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        manifest.append({
            "path": str(p.relative_to(OUTPUT_DIR)).replace("\\", "/"),
            "bytes": int(p.stat().st_size),
            "extension": p.suffix.lower(),
        })
(JSON_DIR / "output_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

ZIP_PATH = shutil.make_archive(
    str(OUTPUT_DIR.parent / f"{ACTIVITY_SLUG}_Ultimate_Web_Project_Bundle"),
    "zip",
    root_dir=OUTPUT_DIR
)

required = [
    FIGURES_DIR / "06_elevation_vs_distance.png",
    FIGURES_DIR / "11_gradient_vs_distance.png",
    MAP_PATH,
    DASHBOARD_PATH,
    INTERACTIVE_DASHBOARD_PATH,
    FULL_INTERACTIVE_PATH,
    JSON_DIR / "dashboard_data.json",
    JSON_DIR / "code_walkthrough.json",
    JSON_DIR / "chart_catalog.json",
    REPORT_DIR / "technical_report.md",
    Path(ZIP_PATH),
]
if HAS_TIME:
    required += [
        FIGURES_DIR / "04_distance_vs_time.png",
        FIGURES_DIR / "08_speed_vs_time.png",
        FIGURES_DIR / "10_acceleration_vs_time.png",
        FIGURES_DIR / "12_speed_and_gradient_vs_time.png",
    ]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError("Missing required outputs:\n" + "\n".join(missing))

print("ALL REQUIRED OUTPUTS PRESENT")
print("Dashboard:", DASHBOARD_PATH)
print("Map:", MAP_PATH)
print("Technical report:", REPORT_PATH)
print("SQLite:", DB_PATH)
print("ZIP:", ZIP_PATH)
print("Files in bundle:", len(manifest))

display(pd.DataFrame(manifest).head(12))
print(f"... {max(0, len(manifest)-12)} more files in the bundle")


# COLAB HANDOFF
from google.colab import files as colab_files
from IPython.display import display, HTML

display(HTML(
    f"""
    <div style="
        margin-top:18px;
        padding:14px 16px;
        border:1px solid #bbf7d0;
        border-radius:10px;
        background:#f0fdf4;
        line-height:1.5;
    ">
      <strong>ZIP ready</strong><br>
      <code>{ZIP_PATH}</code><br>
      <span style="color:#475569">The download should begin automatically.</span>
    </div>
    """
))

colab_files.download(str(ZIP_PATH))
